# Drishti — training the DR grader

Trains the ordinal (CORAL) diabetic-retinopathy grader on real fundus corpora
and exports `grader.onnx` for the API to serve.

**Self-contained.** No clone, no pip install of the project, no repository
access. Run the cells top to bottom.

---

## Before you run

**1. Attach the data.** Right panel → *Add Input* → search and attach:

| Dataset | Kaggle slug | Notes |
|---|---|---|
| APTOS 2019 | `aptos2019-blindness-detection` | Competition data — accept the rules first |
| EyePACS | `diabetic-retinopathy-detection` | 88k images, ~88 GB. Optional for a first run |
| IDRiD | upload as a private dataset | Not redistributable; needed for the lesion benchmark |
| Messidor-2 | upload as a private dataset | Keep as external validation |

Start with **APTOS alone** — it trains in roughly an hour and tells you whether
everything works before you commit to EyePACS.

**2. GPU on.** Settings → Accelerator → *GPU P100* (or T4 x2).

**3. Internet on.** Needed for ImageNet weights, which are not optional here —
see the note above the training cell.

Nothing is downloaded: Kaggle mounts the corpora read-only under
`/kaggle/input`, and the loaders resolve them from there automatically.

## 1 · Environment

In [ ]:
!pip install -q timm onnx 2>&1 | tail -2

import torch, timm
print("torch", torch.__version__)
print("timm ", timm.__version__)
if torch.cuda.is_available():
    print("gpu  ", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("gpu   NONE — enable the accelerator in Settings, or this will take days")

## 2 · Project source

In [ ]:
# The project source, embedded verbatim. Written to disk rather than cloned:
# the repository is private, so a Kaggle kernel cannot reach it, and embedding
# the real source means this notebook cannot drift from the tested code.
import json, sys, os
from pathlib import Path

PROJECT = Path("/kaggle/working/drishti") if Path("/kaggle").exists() else Path("./drishti_pkg")
_SOURCES = json.loads(r"""{
"dr/__init__.py": "",
"dr/metrics.py": "\"\"\"Metrics for DR grading.\n\nQuadratic weighted kappa is the field-standard metric for this task (it is what\nboth the EyePACS and APTOS challenges were scored on) because it is the only\ncommon metric that understands that DR grades are ORDINAL: confusing grade 0\nwith grade 4 is a far worse error than confusing 3 with 4, and plain accuracy\nscores those identically.\n\nEverything here is pure numpy so it can be unit-tested without a dataset, a\nGPU, or a trained model.\n\"\"\"\nimport numpy as np\n\nGRADES = 5\nREFERABLE_THRESHOLD = 2      # ICDR >= 2 (moderate NPDR) needs an ophthalmologist\nSIGHT_THREATENING_THRESHOLD = 3\n\n\ndef confusion(y_true, y_pred, n=GRADES):\n    m = np.zeros((n, n), dtype=np.int64)\n    for t, p in zip(np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()):\n        m[int(t), int(p)] += 1\n    return m\n\n\ndef quadratic_weighted_kappa(y_true, y_pred, n=GRADES):\n    \"\"\"Cohen's kappa with quadratic penalties.\n\n    1.0 = perfect, 0.0 = no better than chance agreement, negative = worse than\n    chance. Implemented directly rather than via sklearn so the expected-matrix\n    construction is visible and testable.\n    \"\"\"\n    y_true = np.asarray(y_true).ravel().astype(int)\n    y_pred = np.asarray(y_pred).ravel().astype(int)\n    if y_true.size == 0:\n        return 0.0\n\n    O = confusion(y_true, y_pred, n).astype(np.float64)\n\n    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing=\"ij\")\n    W = ((i - j) ** 2) / ((n - 1) ** 2)\n\n    # Expected agreement under independence, scaled to the same total as O.\n    hist_true = np.bincount(y_true, minlength=n).astype(np.float64)\n    hist_pred = np.bincount(y_pred, minlength=n).astype(np.float64)\n    E = np.outer(hist_true, hist_pred)\n    if E.sum() == 0:\n        return 0.0\n    E = E * (O.sum() / E.sum())\n\n    denom = (W * E).sum()\n    if denom < 1e-12:\n        # Happens when every label and prediction is the same single class.\n        return 1.0 if (O.trace() == O.sum()) else 0.0\n    return float(1.0 - (W * O).sum() / denom)\n\n\ndef binary_screening_metrics(y_true, y_pred, threshold=REFERABLE_THRESHOLD):\n    \"\"\"Sensitivity / specificity at a severity cut-point.\n\n    This is what a screening programme is actually accountable for. The UK NHS\n    DR screening standard is >=85% sensitivity and >=80% specificity for\n    referable disease; those are the numbers to beat, not accuracy.\n    \"\"\"\n    t = np.asarray(y_true).ravel() >= threshold\n    p = np.asarray(y_pred).ravel() >= threshold\n    tp = int((t & p).sum())\n    fp = int((~t & p).sum())\n    tn = int((~t & ~p).sum())\n    fn = int((t & ~p).sum())\n    sens = tp / max(tp + fn, 1)\n    spec = tn / max(tn + fp, 1)\n    ppv = tp / max(tp + fp, 1)\n    npv = tn / max(tn + fn, 1)\n    return {\"sensitivity\": sens, \"specificity\": spec, \"ppv\": ppv, \"npv\": npv,\n            \"tp\": tp, \"fp\": fp, \"tn\": tn, \"fn\": fn}\n\n\ndef summarise(y_true, y_pred):\n    y_true = np.asarray(y_true).ravel().astype(int)\n    y_pred = np.asarray(y_pred).ravel().astype(int)\n    ref = binary_screening_metrics(y_true, y_pred, REFERABLE_THRESHOLD)\n    stg = binary_screening_metrics(y_true, y_pred, SIGHT_THREATENING_THRESHOLD)\n    return {\n        \"n\": int(y_true.size),\n        \"qwk\": quadratic_weighted_kappa(y_true, y_pred),\n        \"accuracy\": float((y_true == y_pred).mean()) if y_true.size else 0.0,\n        \"within_one\": float((np.abs(y_true - y_pred) <= 1).mean()) if y_true.size else 0.0,\n        \"referable_sensitivity\": ref[\"sensitivity\"],\n        \"referable_specificity\": ref[\"specificity\"],\n        \"referable_ppv\": ref[\"ppv\"],\n        \"sight_threatening_sensitivity\": stg[\"sensitivity\"],\n        \"sight_threatening_specificity\": stg[\"specificity\"],\n        \"confusion\": confusion(y_true, y_pred).tolist(),\n    }\n\n\ndef format_report(stats, title=\"grading\"):\n    lines = [f\"{title}  (n={stats['n']})\",\n             f\"  QWK                          : {stats['qwk']:.4f}\",\n             f\"  exact accuracy               : {stats['accuracy']:.4f}\",\n             f\"  within +/-1 grade            : {stats['within_one']:.4f}\",\n             f\"  referable (>=2) sens / spec  : \"\n             f\"{stats['referable_sensitivity']:.4f} / {stats['referable_specificity']:.4f}\",\n             f\"  sight-threatening (>=3) s/s  : \"\n             f\"{stats['sight_threatening_sensitivity']:.4f} / \"\n             f\"{stats['sight_threatening_specificity']:.4f}\",\n             \"  confusion (rows = true):\"]\n    cm = np.array(stats[\"confusion\"])\n    lines.append(\"        \" + \" \".join(f\"{i:6d}\" for i in range(cm.shape[1])))\n    for i, row in enumerate(cm):\n        lines.append(f\"  true {i}: \" + \" \".join(f\"{v:6d}\" for v in row))\n    return \"\\n\".join(lines)\n\n\n# --------------------------------------------------------------- thresholds\nMIN_THRESHOLD_GAP = 0.15\n\n\ndef optimise_thresholds(y_true, y_score, init=(0.5, 1.5, 2.5, 3.5),\n                        min_gap=MIN_THRESHOLD_GAP):\n    \"\"\"Fit cut-points that convert a continuous severity score into grades.\n\n    A regression head trained on DR outputs something like 2.37; turning that\n    into a grade with plain rounding assumes the classes are evenly spaced on\n    the score axis, which they are not -- the datasets are dominated by grade 0,\n    which drags scores down and systematically under-grades disease. Fitting the\n    cut-points to maximise QWK typically adds several points of kappa and, more\n    importantly, recovers sensitivity on the rare severe grades.\n\n    Coordinate ascent: each boundary is swept in turn while the others are held,\n    which is stable and needs no gradient.\n\n    `min_gap` is not cosmetic. On a small or easy validation split the search\n    happily collapses all four cut-points into a near-identical cluster\n    (e.g. [1.995, 1.996, 1.997, 1.998]), which maximises kappa on THAT split\n    while making grades 1, 2 and 3 unreachable for any future patient -- the\n    model could then only ever output 0 or 4. Forcing a minimum separation\n    keeps every grade addressable.\n    \"\"\"\n    y_true = np.asarray(y_true).ravel().astype(int)\n    y_score = np.asarray(y_score).ravel().astype(float)\n    if y_score.size == 0:\n        return np.array(init, dtype=float), 0.0\n    th = np.array(init, dtype=float)\n\n    def evaluate(candidate):\n        return quadratic_weighted_kappa(y_true, apply_thresholds(y_score, candidate))\n\n    best = evaluate(th)\n    lo_bound = float(y_score.min()) - 0.5\n    hi_bound = float(y_score.max()) + 0.5\n\n    for _ in range(12):\n        improved = False\n        for k in range(len(th)):\n            lo = th[k - 1] + min_gap if k > 0 else lo_bound\n            hi = th[k + 1] - min_gap if k < len(th) - 1 else hi_bound\n            if hi <= lo:\n                continue\n            for cand in np.linspace(lo, hi, 60):\n                trial = th.copy()\n                trial[k] = cand\n                score = evaluate(trial)\n                if score > best + 1e-9:\n                    best, th, improved = score, trial, True\n        if not improved:\n            break\n\n    th = enforce_threshold_separation(th, min_gap)\n    return th, evaluate(th)\n\n\ndef enforce_threshold_separation(thresholds, min_gap=MIN_THRESHOLD_GAP):\n    \"\"\"Push cut-points apart so every grade stays reachable.\"\"\"\n    th = np.sort(np.asarray(thresholds, dtype=float)).copy()\n    for k in range(1, len(th)):\n        if th[k] - th[k - 1] < min_gap:\n            th[k] = th[k - 1] + min_gap\n    return th\n\n\ndef apply_thresholds(y_score, thresholds):\n    \"\"\"Map continuous scores to integer grades via ordered cut-points.\"\"\"\n    y_score = np.asarray(y_score, dtype=float).ravel()\n    out = np.zeros_like(y_score, dtype=np.int64)\n    for t in np.sort(np.asarray(thresholds, dtype=float)):\n        out += (y_score > t).astype(np.int64)\n    return np.clip(out, 0, GRADES - 1)\n\n\ndef coral_expected_grade(logits):\n    \"\"\"Continuous expected grade from CORAL cumulative logits.\n\n    For a non-negative integer variable, E[y] = sum_k P(y > k), which is exactly\n    the sum of the cumulative units. This is the QWK-optimal point estimate:\n    kappa penalises SQUARED distance, and the mean is what minimises squared\n    error -- the mode (argmax of the distribution) does not, and on a broad or\n    bimodal posterior the two can differ by two whole grades.\n\n    Returned unrounded so thresholds can be fitted against it.\n    \"\"\"\n    dist = coral_logits_to_distribution(logits)\n    if dist.ndim == 1:\n        dist = dist[None, :]\n    return (dist * np.arange(GRADES)).sum(axis=1)\n\n\ndef coral_logits_to_grade(logits):\n    \"\"\"Decode CORAL ordinal logits into an integer grade.\n\n    Derived from the same distribution the report displays, so the headline\n    grade can never disagree with the probability bar shown beside it. Using\n    the raw sigmoid sum here instead would let the two drift apart whenever the\n    network emits non-monotone cumulative outputs.\n    \"\"\"\n    return np.clip(np.round(coral_expected_grade(logits)).astype(int), 0, GRADES - 1)\n\n\ndef coral_logits_to_distribution(logits):\n    \"\"\"Turn cumulative logits into a proper 5-way probability distribution.\n\n    P(y = k) = P(y > k-1) - P(y > k), with the cumulative probabilities forced\n    to be non-increasing first so no class can come out negative.\n    \"\"\"\n    probs = 1.0 / (1.0 + np.exp(-np.asarray(logits, dtype=float)))\n    single = probs.ndim == 1\n    if single:\n        probs = probs[None, :]\n    n, k = probs.shape\n    # Enforce monotonicity: P(y>0) >= P(y>1) >= ...\n    probs = np.minimum.accumulate(probs, axis=1)\n    cum = np.concatenate([np.ones((n, 1)), probs, np.zeros((n, 1))], axis=1)\n    dist = cum[:, :-1] - cum[:, 1:]\n    dist = np.clip(dist, 0.0, None)\n    dist /= np.maximum(dist.sum(axis=1, keepdims=True), 1e-12)\n    return dist[0] if single else dist\n",
"dr/datasets.py": "\"\"\"Unified access to the four public DR corpora.\n\nEvery loader returns the same record shape, so training, evaluation and the\nlesion benchmark never need to know which corpus they are looking at:\n\n    {\"image_path\": str, \"grade\": int 0-4, \"patient_id\": str, \"eye\": \"L\"|\"R\"|None,\n     \"dataset\": str, \"split\": str}\n\nCorpora and what each is for:\n\n  IDRiD      516 graded images, and 81 with PIXEL-LEVEL lesion masks\n             (microaneurysm, haemorrhage, hard exudate, soft exudate). The only\n             public set that can score the morphological segmenter's \"where\"\n             channel, which is why it is worth its small size.\n  APTOS-2019 3,662 images, ICDR 0-4, Indian population (Aravind Eye Hospital).\n             The closest public proxy to the deployment population.\n  EyePACS    88,702 images, ICDR 0-4. Large, noisy, heavily imbalanced; the\n             corpus a CNN needs to actually generalise.\n  Messidor-2 1,748 images. Held out as an EXTERNAL validation set and never\n             trained on, so the reported number is not a within-corpus score.\n\nPaths are resolved through `DR_DATA_ROOT` (env var) or an explicit root, so the\nsame code runs on Kaggle (/kaggle/input/...) and locally without edits.\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport os\nimport re\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n# Kaggle mounts each dataset read-only under /kaggle/input/<slug>.\nKAGGLE_INPUT = Path(\"/kaggle/input\")\n\nIMAGE_SUFFIXES = {\".jpg\", \".jpeg\", \".png\", \".tif\", \".tiff\", \".JPG\", \".JPEG\", \".PNG\"}\n\nLESION_CLASSES = {\n    \"MA\": \"Microaneurysms\",\n    \"HEM\": \"Haemorrhages\",\n    \"EX\": \"Hard Exudates\",\n    \"CWS\": \"Soft Exudates\",\n}\n\n\n@dataclass\nclass Record:\n    image_path: str\n    grade: int\n    patient_id: str\n    dataset: str\n    eye: str | None = None\n    split: str = \"train\"\n    lesion_masks: dict = field(default_factory=dict)   # class -> mask path\n\n    def as_dict(self):\n        return {\"image_path\": self.image_path, \"grade\": int(self.grade),\n                \"patient_id\": self.patient_id, \"dataset\": self.dataset,\n                \"eye\": self.eye, \"split\": self.split,\n                \"lesion_masks\": dict(self.lesion_masks)}\n\n\nclass DatasetUnavailable(RuntimeError):\n    \"\"\"Raised when a corpus is not present on disk.\n\n    Carries the download instructions rather than just failing, because the\n    single most common way this pipeline breaks for a new user is a dataset\n    that was never fetched.\n    \"\"\"\n\n\ndef data_root(explicit=None) -> Path:\n    if explicit:\n        return Path(explicit)\n    env = os.environ.get(\"DR_DATA_ROOT\")\n    if env:\n        return Path(env)\n    if KAGGLE_INPUT.exists():\n        return KAGGLE_INPUT\n    return Path(__file__).resolve().parent.parent / \"datasets\"\n\n\ndef _first_existing(root: Path, candidates):\n    for c in candidates:\n        p = root / c\n        if p.exists():\n            return p\n    return None\n\n\ndef _index_images(folder: Path):\n    \"\"\"Map bare stem -> path, so a CSV id can be resolved regardless of the\n    extension or the nesting the corpus happens to use.\"\"\"\n    out = {}\n    for p in folder.rglob(\"*\"):\n        if p.suffix in IMAGE_SUFFIXES and p.is_file():\n            out.setdefault(p.stem, p)\n    return out\n\n\ndef _read_csv(path: Path):\n    with open(path, newline=\"\", encoding=\"utf-8-sig\") as fh:\n        return list(csv.DictReader(fh))\n\n\ndef _col(row, *names):\n    \"\"\"Fetch a column by any of several spellings; corpora disagree on case\n    and separators (`Retinopathy grade`, `diagnosis`, `level`, ...).\"\"\"\n    norm = {re.sub(r\"[^a-z0-9]\", \"\", k.lower()): v for k, v in row.items() if k}\n    for n in names:\n        key = re.sub(r\"[^a-z0-9]\", \"\", n.lower())\n        if key in norm:\n            return norm[key]\n    return None\n\n\n# --------------------------------------------------------------- APTOS 2019\ndef load_aptos(root=None, split=\"train\"):\n    root = data_root(root)\n    base = _first_existing(root, [\"aptos2019-blindness-detection\", \"aptos2019\",\n                                  \"aptos\", \"APTOS2019\"])\n    if base is None:\n        raise DatasetUnavailable(\n            \"APTOS 2019 not found. On Kaggle add the dataset \"\n            \"'aptos2019-blindness-detection'; locally run\\n\"\n            \"  kaggle competitions download -c aptos2019-blindness-detection\\n\"\n            f\"and unpack under {root}/aptos2019-blindness-detection\")\n\n    csv_path = _first_existing(base, [\"train.csv\", \"train_1.csv\"])\n    img_dir = _first_existing(base, [\"train_images\", \"train\"])\n    if csv_path is None or img_dir is None:\n        raise DatasetUnavailable(f\"APTOS present at {base} but train.csv/train_images missing\")\n\n    index = _index_images(img_dir)\n    records = []\n    for row in _read_csv(csv_path):\n        ident = _col(row, \"id_code\", \"image\", \"id\")\n        grade = _col(row, \"diagnosis\", \"level\", \"grade\")\n        if ident is None or grade is None:\n            continue\n        path = index.get(str(ident))\n        if path is None:\n            continue\n        # APTOS ships one image per patient with no laterality metadata, so the\n        # image id IS the grouping key. Treating each image as its own patient\n        # is correct here and must not be copied to corpora where it is not.\n        records.append(Record(str(path), int(grade), f\"aptos:{ident}\",\n                              \"aptos\", None, split))\n    return records\n\n\n# ------------------------------------------------------------------ EyePACS\ndef load_eyepacs(root=None, split=\"train\"):\n    root = data_root(root)\n    base = _first_existing(root, [\"diabetic-retinopathy-detection\", \"eyepacs\",\n                                  \"diabetic-retinopathy-resized\"])\n    if base is None:\n        raise DatasetUnavailable(\n            \"EyePACS not found. On Kaggle add 'diabetic-retinopathy-detection' \"\n            \"(or the resized mirror 'diabetic-retinopathy-resized'); locally\\n\"\n            \"  kaggle competitions download -c diabetic-retinopathy-detection\")\n\n    csv_path = _first_existing(base, [\"trainLabels.csv\", \"trainLabels/trainLabels.csv\",\n                                      \"train.csv\", \"trainLabels_cropped.csv\"])\n    img_dir = _first_existing(base, [\"train\", \"resized_train\", \"train_images\",\n                                     \"resized_train_cropped\"])\n    if csv_path is None or img_dir is None:\n        raise DatasetUnavailable(f\"EyePACS present at {base} but labels/images missing\")\n\n    index = _index_images(img_dir)\n    records = []\n    for row in _read_csv(csv_path):\n        ident = _col(row, \"image\", \"id_code\")\n        grade = _col(row, \"level\", \"diagnosis\", \"grade\")\n        if ident is None or grade is None:\n            continue\n        path = index.get(str(ident))\n        if path is None:\n            continue\n        # EyePACS ids are '<patient>_<left|right>'. Both eyes of one patient are\n        # strongly correlated, so the patient -- not the image -- is the unit\n        # that must stay on one side of a split.\n        m = re.match(r\"^(\\d+)_(left|right)$\", str(ident))\n        pid = f\"eyepacs:{m.group(1)}\" if m else f\"eyepacs:{ident}\"\n        eye = {\"left\": \"L\", \"right\": \"R\"}.get(m.group(2)) if m else None\n        records.append(Record(str(path), int(grade), pid, \"eyepacs\", eye, split))\n    return records\n\n\n# -------------------------------------------------------------------- IDRiD\ndef load_idrid(root=None, split=\"train\", with_masks=True):\n    \"\"\"IDRiD grading set, with lesion masks attached where they exist.\"\"\"\n    root = data_root(root)\n    base = _first_existing(root, [\"idrid\", \"IDRiD\", \"indian-diabetic-retinopathy-image-dataset\",\n                                  \"diabetic-retinopathy-segmentation\"])\n    if base is None:\n        raise DatasetUnavailable(\n            \"IDRiD not found. Download from \"\n            \"https://idrid.grand-challenge.org/ (free registration) and unpack \"\n            f\"under {root}/idrid, keeping the 'A. Segmentation' and \"\n            \"'B. Disease Grading' folders.\")\n\n    grade_csv = None\n    for p in base.rglob(\"*.csv\"):\n        name = p.name.lower()\n        if \"groundtruth\" in name.replace(\" \", \"\") or \"grading\" in name or \"label\" in name:\n            if \"train\" in name or split == \"train\":\n                grade_csv = p\n                break\n    if grade_csv is None:\n        raise DatasetUnavailable(f\"IDRiD found at {base} but no grading CSV located\")\n\n    grading_imgs = {}\n    for d in base.rglob(\"*\"):\n        if d.is_dir() and \"grading\" in d.name.lower().replace(\" \", \"\"):\n            grading_imgs.update(_index_images(d))\n    if not grading_imgs:\n        grading_imgs = _index_images(base)\n\n    masks_by_stem = _idrid_masks(base) if with_masks else {}\n\n    records = []\n    for row in _read_csv(grade_csv):\n        ident = _col(row, \"Image name\", \"image\", \"id\")\n        grade = _col(row, \"Retinopathy grade\", \"retinopathygrade\", \"grade\", \"level\")\n        if ident is None or grade is None:\n            continue\n        stem = str(ident).strip()\n        path = grading_imgs.get(stem)\n        if path is None:\n            continue\n        records.append(Record(str(path), int(grade), f\"idrid:{stem}\", \"idrid\",\n                              None, split, masks_by_stem.get(stem, {})))\n    return records\n\n\ndef _idrid_masks(base: Path):\n    \"\"\"Locate per-class lesion masks.\n\n    IDRiD names them '<stem>_MA.tif', '<stem>_HE.tif', '<stem>_EX.tif',\n    '<stem>_SE.tif' inside folders whose names carry the class. Both the\n    filename suffix and the parent folder are checked, because the two IDRiD\n    distributions on Kaggle differ in which one they preserve.\n    \"\"\"\n    suffix_map = {\"ma\": \"MA\", \"he\": \"HEM\", \"ex\": \"EX\", \"se\": \"CWS\"}\n    folder_map = {\"microaneurysm\": \"MA\", \"haemorrhage\": \"HEM\", \"hemorrhage\": \"HEM\",\n                  \"hardexudate\": \"EX\", \"softexudate\": \"CWS\", \"opticdisc\": None}\n    out = {}\n    for p in base.rglob(\"*\"):\n        if p.suffix.lower() not in {\".tif\", \".tiff\", \".png\"} or not p.is_file():\n            continue\n        m = re.match(r\"^(.*?)_([A-Za-z]{2})$\", p.stem)\n        cls = None\n        if m and m.group(2).lower() in suffix_map:\n            stem, cls = m.group(1), suffix_map[m.group(2).lower()]\n        else:\n            folder = re.sub(r\"[^a-z]\", \"\", p.parent.name.lower())\n            for key, val in folder_map.items():\n                if key in folder:\n                    stem, cls = p.stem, val\n                    break\n        if cls:\n            out.setdefault(stem, {})[cls] = str(p)\n    return out\n\n\ndef load_idrid_segmentation(root=None):\n    \"\"\"Only the images that carry pixel-level lesion ground truth.\"\"\"\n    return [r for r in load_idrid(root, with_masks=True) if r.lesion_masks]\n\n\n# --------------------------------------------------------------- Messidor-2\ndef load_messidor2(root=None, split=\"external\"):\n    root = data_root(root)\n    base = _first_existing(root, [\"messidor2\", \"messidor-2\", \"Messidor-2\", \"messidor\"])\n    if base is None:\n        raise DatasetUnavailable(\n            \"Messidor-2 not found. Request access at \"\n            \"https://www.adcis.net/en/third-party/messidor2/ and unpack under \"\n            f\"{root}/messidor2 with its grading CSV.\")\n\n    csv_path = None\n    for p in base.rglob(\"*.csv\"):\n        if any(k in p.name.lower() for k in (\"grade\", \"label\", \"diagnos\", \"abnormal\")):\n            csv_path = p\n            break\n    if csv_path is None:\n        raise DatasetUnavailable(f\"Messidor-2 at {base} but no grading CSV found\")\n\n    index = _index_images(base)\n    records = []\n    for row in _read_csv(csv_path):\n        ident = _col(row, \"image_id\", \"image\", \"id\", \"imagename\")\n        grade = _col(row, \"adjudicated_dr_grade\", \"dr_grade\", \"grade\", \"diagnosis\", \"level\")\n        if ident is None or grade in (None, \"\"):\n            continue\n        stem = Path(str(ident)).stem\n        path = index.get(stem)\n        if path is None:\n            continue\n        m = re.match(r\"^(\\d+)_\", stem)\n        pid = f\"messidor2:{m.group(1)}\" if m else f\"messidor2:{stem}\"\n        records.append(Record(str(path), int(float(grade)), pid, \"messidor2\", None, split))\n    return records\n\n\n# ------------------------------------------------------------------ registry\nLOADERS = {\n    \"aptos\": load_aptos,\n    \"eyepacs\": load_eyepacs,\n    \"idrid\": load_idrid,\n    \"messidor2\": load_messidor2,\n}\n\n\ndef load(names, root=None, strict=False):\n    \"\"\"Load and concatenate several corpora by name.\n\n    With strict=False a missing corpus is reported and skipped, so a run that\n    has APTOS but not EyePACS still trains instead of dying at import time.\n    \"\"\"\n    if isinstance(names, str):\n        names = [names]\n    records, missing = [], []\n    for n in names:\n        if n not in LOADERS:\n            raise KeyError(f\"unknown dataset '{n}'; known: {sorted(LOADERS)}\")\n        try:\n            found = LOADERS[n](root)\n            records.extend(found)\n            print(f\"  {n:10} {len(found):>6} images\")\n        except DatasetUnavailable as e:\n            if strict:\n                raise\n            missing.append(f\"{n}: {e}\")\n    for m in missing:\n        print(f\"  SKIPPED {m}\")\n    return records\n\n\ndef grade_histogram(records):\n    h = [0] * 5\n    for r in records:\n        if 0 <= r.grade <= 4:\n            h[r.grade] += 1\n    return h\n\n\ndef describe(records):\n    h = grade_histogram(records)\n    total = max(sum(h), 1)\n    lines = [f\"{len(records)} images, {len({r.patient_id for r in records})} patients\"]\n    for g, c in enumerate(h):\n        lines.append(f\"  grade {g}: {c:>6}  ({c / total:5.1%})\")\n    return \"\\n\".join(lines)\n",
"dr/splits.py": "\"\"\"Patient-grouped, grade-stratified cross-validation splits.\n\nTwo mistakes are easy to make here and both silently inflate the reported\nscore, which is the worst kind of bug in a clinical prototype:\n\n  1. Splitting by IMAGE. EyePACS and Messidor-2 contain both eyes of the same\n     patient, and the two eyes of one diabetic are highly correlated. A random\n     image split puts a patient's left eye in train and the right eye in\n     validation, and the model is scored partly on memorisation. Every split\n     here is grouped by patient.\n\n  2. Ignoring grade imbalance. Grades 3 and 4 are only a few percent of these\n     corpora, so an ungrouped random fold can end up with almost no severe\n     cases and a meaningless sensitivity estimate. Folds are stratified by the\n     patient's worst grade.\n\"\"\"\nfrom __future__ import annotations\n\nfrom collections import defaultdict\n\nimport numpy as np\n\n\ndef patient_groups(records):\n    \"\"\"patient_id -> (indices, worst grade seen for that patient).\"\"\"\n    by_patient = defaultdict(list)\n    for i, r in enumerate(records):\n        by_patient[r.patient_id].append(i)\n    return {pid: (idx, max(records[i].grade for i in idx))\n            for pid, idx in by_patient.items()}\n\n\ndef stratified_group_folds(records, n_folds=5, seed=0):\n    \"\"\"Assign every patient to exactly one fold.\n\n    Patients are bucketed by their worst grade and dealt round-robin into\n    folds, which keeps the rare severe grades evenly spread instead of leaving\n    them to chance.\n    \"\"\"\n    groups = patient_groups(records)\n    by_grade = defaultdict(list)\n    for pid, (_, worst) in groups.items():\n        by_grade[worst].append(pid)\n\n    rng = np.random.default_rng(seed)\n    fold_of_patient = {}\n    for grade in sorted(by_grade):\n        pids = by_grade[grade]\n        rng.shuffle(pids)\n        # Offset the starting fold per grade so small strata do not all pile\n        # into fold 0.\n        offset = rng.integers(0, n_folds)\n        for k, pid in enumerate(pids):\n            fold_of_patient[pid] = int((k + offset) % n_folds)\n\n    folds = [[] for _ in range(n_folds)]\n    for pid, (idx, _) in groups.items():\n        folds[fold_of_patient[pid]].extend(idx)\n    return [sorted(f) for f in folds]\n\n\ndef train_val_split(records, val_fraction=0.2, seed=0):\n    n_folds = max(2, int(round(1.0 / max(val_fraction, 1e-6))))\n    folds = stratified_group_folds(records, n_folds=n_folds, seed=seed)\n    val = folds[0]\n    train = [i for f in folds[1:] for i in f]\n    return sorted(train), sorted(val)\n\n\ndef assert_no_patient_leakage(records, train_idx, val_idx):\n    \"\"\"Fail loudly rather than quietly reporting an inflated score.\"\"\"\n    tr = {records[i].patient_id for i in train_idx}\n    va = {records[i].patient_id for i in val_idx}\n    overlap = tr & va\n    if overlap:\n        raise AssertionError(\n            f\"{len(overlap)} patient(s) appear in both train and validation, \"\n            f\"e.g. {sorted(overlap)[:5]}. The reported score would be inflated.\")\n\n\ndef class_balanced_weights(records, indices=None, power=0.5):\n    \"\"\"Per-sample weights for a WeightedRandomSampler.\n\n    EyePACS is roughly 73% grade 0 and under 3% grade 4. Training on the raw\n    distribution produces a model that is excellent at saying \"healthy\" and\n    close to useless on the grades that cost people their sight.\n\n    Full inverse-frequency weighting (power=1.0) over-corrects: grade-4 images\n    then repeat so often the model memorises them. power=0.5 (inverse sqrt\n    frequency) is the usual compromise and is the default here.\n    \"\"\"\n    indices = list(range(len(records))) if indices is None else list(indices)\n    counts = np.zeros(5, dtype=np.float64)\n    for i in indices:\n        counts[records[i].grade] += 1\n    freq = np.maximum(counts, 1.0)\n    w = (freq.sum() / freq) ** power\n    w = w / w.sum() * 5.0                      # keep weights near 1.0 on average\n    return np.array([w[records[i].grade] for i in indices], dtype=np.float64)\n\n\ndef summarise_split(records, train_idx, val_idx):\n    def hist(idx):\n        h = np.zeros(5, dtype=int)\n        for i in idx:\n            h[records[i].grade] += 1\n        return h\n    th, vh = hist(train_idx), hist(val_idx)\n    lines = [\"         \" + \" \".join(f\"{g:>7}\" for g in range(5)) + \"     total\",\n             \"  train: \" + \" \".join(f\"{v:>7}\" for v in th) + f\" {th.sum():>9}\",\n             \"  val  : \" + \" \".join(f\"{v:>7}\" for v in vh) + f\" {vh.sum():>9}\"]\n    return \"\\n\".join(lines)\n",
"dr/transforms.py": "\"\"\"Fundus preprocessing and augmentation, in plain OpenCV.\n\nWritten directly against cv2 rather than pulling in albumentations for two\nreasons: the transforms that matter here are fundus-specific (field-of-view\ncropping, Graham normalisation) and not in any generic library, and keeping the\ndependency list short means the same code runs unmodified on Kaggle, in the\nDocker image, and on a CPU-only PHC edge box.\n\nThe single most important step is the FOV crop. Public DR corpora are a mess of\nletterboxed, off-centre, differently-zoomed captures; without normalising the\nretinal circle first, a network spends its capacity learning which hospital\ntook the photograph.\n\"\"\"\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nIMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)\nIMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)\n\n\n# --------------------------------------------------------------- FOV crop\ndef fov_bbox(bgr, threshold_scale=0.06):\n    \"\"\"Bounding box of the illuminated retinal circle.\n\n    Thresholds relative to the image's own maximum rather than at a fixed grey\n    level, so an underexposed capture is cropped as tightly as a bright one.\n    \"\"\"\n    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n    small = cv2.resize(gray, (256, 256), interpolation=cv2.INTER_AREA)\n    thr = max(6.0, float(small.max()) * threshold_scale)\n    mask = (small > thr).astype(np.uint8)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,\n                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if n <= 1:\n        return 0, 0, bgr.shape[1], bgr.shape[0]\n    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    sy, sx = bgr.shape[0] / 256.0, bgr.shape[1] / 256.0\n    x = int(stats[k, cv2.CC_STAT_LEFT] * sx)\n    y = int(stats[k, cv2.CC_STAT_TOP] * sy)\n    w = int(stats[k, cv2.CC_STAT_WIDTH] * sx)\n    h = int(stats[k, cv2.CC_STAT_HEIGHT] * sy)\n    return x, y, max(w, 1), max(h, 1)\n\n\ndef crop_to_fov(bgr, pad=0.02):\n    x, y, w, h = fov_bbox(bgr)\n    side = int(max(w, h) * (1 + pad))\n    cx, cy = x + w // 2, y + h // 2\n    half = side // 2\n    p = side\n    padded = cv2.copyMakeBorder(bgr, p, p, p, p, cv2.BORDER_CONSTANT, value=(0, 0, 0))\n    return padded[cy - half + p:cy + half + p, cx - half + p:cx + half + p]\n\n\ndef circular_mask(size, shrink=0.97):\n    m = np.zeros((size, size), np.uint8)\n    cv2.circle(m, (size // 2, size // 2), int(size / 2 * shrink), 1, -1)\n    return m\n\n\ndef graham_normalise(bgr, size, sigma_frac=1 / 30.0, weight=4.0, bias=128.0):\n    \"\"\"Ben Graham's EyePACS-winning normalisation.\n\n    Subtracts a heavily blurred copy of the image, which removes the\n    camera-specific colour cast and vignetting while amplifying exactly the\n    local contrast that microaneurysms and exudates live in. It remains the\n    strongest single preprocessing step for this task.\n    \"\"\"\n    blur = cv2.GaussianBlur(bgr, (0, 0), size * sigma_frac)\n    out = cv2.addWeighted(bgr, weight, blur, -weight, bias)\n    mask = circular_mask(out.shape[0])\n    return cv2.bitwise_and(out, out, mask=mask)\n\n\ndef load_and_prepare(path, size=512, graham=True):\n    \"\"\"Disk -> normalised square BGR uint8. Used identically at train and serve\n    time; any divergence between the two is a silent accuracy loss.\"\"\"\n    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)\n    if bgr is None:\n        raise ValueError(f\"could not read image: {path}\")\n    bgr = crop_to_fov(bgr)\n    if bgr.size == 0:\n        raise ValueError(f\"empty crop for image: {path}\")\n    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)\n    if graham:\n        bgr = graham_normalise(bgr, size)\n    else:\n        bgr = cv2.bitwise_and(bgr, bgr, mask=circular_mask(size))\n    return bgr\n\n\n# ------------------------------------------------------------ augmentation\ndef augment(bgr, rng):\n    \"\"\"Geometric and photometric jitter appropriate to fundus photography.\n\n    Retinal images have no canonical orientation once laterality is discarded,\n    so full flips and rotations are safe and are the highest-value augmentation\n    here. Photometric jitter is deliberately mild: push brightness or contrast\n    too far and genuine microaneurysms are destroyed, which teaches the network\n    to ignore the smallest and earliest sign of disease.\n    \"\"\"\n    h, w = bgr.shape[:2]\n\n    if rng.random() < 0.5:\n        bgr = cv2.flip(bgr, 1)\n    if rng.random() < 0.5:\n        bgr = cv2.flip(bgr, 0)\n\n    angle = rng.uniform(0, 360)\n    scale = rng.uniform(0.92, 1.08)\n    tx = rng.uniform(-0.02, 0.02) * w\n    ty = rng.uniform(-0.02, 0.02) * h\n    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)\n    M[0, 2] += tx\n    M[1, 2] += ty\n    bgr = cv2.warpAffine(bgr, M, (w, h), flags=cv2.INTER_LINEAR,\n                         borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))\n\n    if rng.random() < 0.7:\n        alpha = rng.uniform(0.9, 1.1)          # contrast\n        beta = rng.uniform(-10, 10)            # brightness\n        bgr = cv2.convertScaleAbs(bgr, alpha=alpha, beta=beta)\n\n    if rng.random() < 0.3:\n        gamma = rng.uniform(0.85, 1.15)\n        lut = np.clip(((np.arange(256) / 255.0) ** (1.0 / gamma)) * 255, 0, 255)\n        bgr = cv2.LUT(bgr, lut.astype(np.uint8))\n\n    if rng.random() < 0.2:\n        bgr = cv2.GaussianBlur(bgr, (0, 0), rng.uniform(0.4, 1.0))\n\n    return cv2.bitwise_and(bgr, bgr, mask=circular_mask(h))\n\n\ndef to_tensor(bgr):\n    \"\"\"BGR uint8 HWC -> normalised RGB float CHW, as a numpy array.\n\n    Returned as numpy rather than torch so this module stays importable without\n    torch (the edge inference path uses onnxruntime and no torch at all).\n    \"\"\"\n    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0\n    rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD\n    return np.ascontiguousarray(rgb.transpose(2, 0, 1))\n",
"dr/model.py": "\"\"\"Grading network: timm backbone + CORAL ordinal head.\n\nWhy ordinal rather than 5-way softmax:\n\nA softmax head treats the five ICDR grades as unrelated categories, so\npredicting 0 for a grade-4 eye costs it exactly what predicting 3 costs. That\nis wrong clinically and wrong for the metric the task is scored on (QWK\npenalises squared distance). CORAL (Cao et al., 2020) instead learns K-1\ncumulative units -- \"is the grade > 0?\", \"> 1?\", \"> 2?\", \"> 3?\" -- sharing one\nfeature vector and one weight vector, differing only in a per-unit bias. That\nshared weight is what guarantees the predicted cumulative probabilities stay\nmonotonic, so the model can never assert P(grade>2) > P(grade>1).\n\nIt also gives the triage layer something a softmax cannot: a calibrated\nP(grade >= 2), which is precisely the referral decision.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nGRADES = 5\n\n\nclass CoralHead(nn.Module):\n    \"\"\"K-1 cumulative logits from one shared projection plus per-unit biases.\"\"\"\n\n    def __init__(self, in_features, num_classes=GRADES):\n        super().__init__()\n        self.fc = nn.Linear(in_features, 1, bias=False)\n        # Biases initialised in decreasing order so the head starts out\n        # monotone and ordered rather than having to learn the ordering.\n        self.bias = nn.Parameter(torch.linspace(1.5, -1.5, num_classes - 1))\n\n    def forward(self, x):\n        return self.fc(x) + self.bias          # (N, K-1)\n\n\ndef coral_targets(grades, num_classes=GRADES):\n    \"\"\"Grade -> binary cumulative targets. Grade 3 becomes [1, 1, 1, 0].\"\"\"\n    levels = torch.arange(num_classes - 1, device=grades.device)[None, :]\n    return (grades[:, None] > levels).float()\n\n\nclass CoralLoss(nn.Module):\n    \"\"\"Binary cross-entropy over the cumulative units.\n\n    `importance` lets the sight-threatening boundaries carry more weight than\n    the 0-vs-1 boundary. That is a deliberate clinical choice: separating \"no\n    retinopathy\" from \"mild\" is the least consequential decision the model\n    makes, while the >=2 boundary is the referral itself.\n    \"\"\"\n\n    def __init__(self, importance=(1.0, 1.6, 1.6, 1.4)):\n        super().__init__()\n        self.register_buffer(\"importance\", torch.tensor(importance, dtype=torch.float32))\n\n    def forward(self, logits, grades):\n        targets = coral_targets(grades, logits.shape[1] + 1)\n        per_unit = nn.functional.binary_cross_entropy_with_logits(\n            logits, targets, reduction=\"none\")\n        return (per_unit * self.importance[None, :]).mean()\n\n\nclass DRGrader(nn.Module):\n    def __init__(self, backbone=\"tf_efficientnet_b3_ns\", pretrained=True,\n                 drop_rate=0.3, num_classes=GRADES, in_chans=3):\n        super().__init__()\n        import timm\n        self.backbone_name = backbone\n        self.encoder = timm.create_model(\n            backbone, pretrained=pretrained, num_classes=0,\n            drop_rate=drop_rate, in_chans=in_chans)\n        self.head = CoralHead(self.encoder.num_features, num_classes)\n        self.num_classes = num_classes\n\n    def forward(self, x):\n        return self.head(self.encoder(x))\n\n    @torch.no_grad()\n    def predict(self, x):\n        \"\"\"Cumulative logits -> (expected grade, 5-way distribution).\"\"\"\n        from .metrics import coral_expected_grade, coral_logits_to_distribution\n        logits = self(x).float().cpu().numpy()\n        return coral_expected_grade(logits), coral_logits_to_distribution(logits)\n\n    def gradcam_layer(self):\n        \"\"\"Last convolutional stage, for Grad-CAM.\n\n        timm backbones expose their stages under different attribute names, so\n        the final 4-D-output module is located by inspection rather than by\n        hard-coding a path that silently breaks when the backbone changes.\n        \"\"\"\n        candidates = [m for m in self.encoder.modules()\n                      if isinstance(m, (nn.Conv2d, nn.BatchNorm2d))]\n        if not candidates:\n            raise RuntimeError(\"no convolutional layer found for Grad-CAM\")\n        return candidates[-1]\n\n\ndef build(backbone=\"tf_efficientnet_b3_ns\", pretrained=True, **kw):\n    return DRGrader(backbone=backbone, pretrained=pretrained, **kw)\n\n\n# ------------------------------------------------------------------- export\ndef export_onnx(model, path, size=512, opset=17):\n    \"\"\"Export for onnxruntime serving.\n\n    The edge/PHC path runs CPU-only with no torch installed at all, and the GPU\n    server uses ONNX for a stable, versioned artefact that does not depend on\n    the training code still being importable.\n    \"\"\"\n    model = model.eval().cpu()\n    dummy = torch.randn(1, 3, size, size)\n    common = dict(input_names=[\"image\"], output_names=[\"cumulative_logits\"],\n                  dynamic_axes={\"image\": {0: \"batch\"},\n                                \"cumulative_logits\": {0: \"batch\"}},\n                  opset_version=opset)\n    try:\n        # torch>=2.5 defaults to the dynamo exporter, which needs onnxscript.\n        # Fall back to the long-stable TorchScript exporter when it is absent,\n        # so a missing optional package cannot cost a finished training run\n        # its deployable artefact.\n        torch.onnx.export(model, dummy, str(path), dynamo=False,\n                          do_constant_folding=True, **common)\n    except TypeError:\n        torch.onnx.export(model, dummy, str(path),\n                          do_constant_folding=True, **common)\n    return path\n\n\nclass OnnxGrader:\n    \"\"\"Inference-only grader with no torch dependency.\"\"\"\n\n    def __init__(self, path, providers=None):\n        import onnxruntime as ort\n        if providers is None:\n            available = ort.get_available_providers()\n            providers = ([p for p in (\"CUDAExecutionProvider\",) if p in available]\n                         + [\"CPUExecutionProvider\"])\n        self.session = ort.InferenceSession(str(path), providers=providers)\n        self.input_name = self.session.get_inputs()[0].name\n\n    def __call__(self, batch):\n        \"\"\"batch: (N,3,H,W) float32 -> (expected grade, distribution).\"\"\"\n        from .metrics import coral_expected_grade, coral_logits_to_distribution\n        logits = self.session.run(None, {self.input_name:\n                                         np.ascontiguousarray(batch, np.float32)})[0]\n        return coral_expected_grade(logits), coral_logits_to_distribution(logits)\n",
"dr/torchdata.py": "\"\"\"Torch Dataset and DataLoader construction.\n\nDecoding and resizing multi-megapixel fundus JPEGs is the real bottleneck in\nthis pipeline -- far more than the GPU forward pass -- so the loader supports a\npre-resized cache. Building that cache once turns an EyePACS epoch from\nhours of JPEG decoding into minutes.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\n\nfrom . import transforms as T\nfrom .splits import class_balanced_weights\n\n\nclass FundusDataset(Dataset):\n    def __init__(self, records, indices=None, size=512, train=False,\n                 graham=True, cache_dir=None, seed=0):\n        self.records = records\n        self.indices = list(range(len(records))) if indices is None else list(indices)\n        self.size = size\n        self.train = train\n        self.graham = graham\n        self.cache_dir = Path(cache_dir) if cache_dir else None\n        if self.cache_dir:\n            self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.seed = seed\n\n    def __len__(self):\n        return len(self.indices)\n\n    def _cache_path(self, rec):\n        # Key on the settings that change the pixels, so switching resolution\n        # or normalisation cannot silently reuse the wrong cache.\n        key = f\"{rec.image_path}|{self.size}|{int(self.graham)}\"\n        return self.cache_dir / (hashlib.md5(key.encode()).hexdigest() + \".png\")\n\n    def _load(self, rec):\n        if self.cache_dir:\n            cp = self._cache_path(rec)\n            if cp.exists():\n                img = cv2.imread(str(cp), cv2.IMREAD_COLOR)\n                if img is not None:\n                    return img\n            img = T.load_and_prepare(rec.image_path, self.size, self.graham)\n            cv2.imwrite(str(cp), img)\n            return img\n        return T.load_and_prepare(rec.image_path, self.size, self.graham)\n\n    def __getitem__(self, i):\n        rec = self.records[self.indices[i]]\n        img = self._load(rec)\n        if self.train:\n            # Seed per (epoch-agnostic) worker+index so augmentation differs\n            # across workers but stays reproducible for a fixed seed.\n            rng = np.random.default_rng((self.seed * 1_000_003 + i) % (2**32))\n            img = T.augment(img, rng)\n        x = torch.from_numpy(T.to_tensor(img))\n        return x, torch.tensor(rec.grade, dtype=torch.long), self.indices[i]\n\n\ndef build_loaders(records, train_idx, val_idx, size=512, batch_size=16,\n                  workers=2, balanced=True, cache_dir=None, seed=0,\n                  balance_power=0.5):\n    train_ds = FundusDataset(records, train_idx, size, True, True, cache_dir, seed)\n    val_ds = FundusDataset(records, val_idx, size, False, True, cache_dir, seed)\n\n    if balanced:\n        w = class_balanced_weights(records, train_idx, power=balance_power)\n        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),\n                                        num_samples=len(train_idx), replacement=True)\n        shuffle = False\n    else:\n        sampler, shuffle = None, True\n\n    common = dict(num_workers=workers, pin_memory=torch.cuda.is_available(),\n                  persistent_workers=workers > 0)\n    train_dl = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,\n                          shuffle=shuffle, drop_last=True, **common)\n    val_dl = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, **common)\n    return train_dl, val_dl\n\n\ndef build_cache(records, size=512, graham=True, cache_dir=None, workers=4):\n    \"\"\"Materialise the resized cache up front.\n\n    Worth running once before training: it removes full-resolution JPEG\n    decoding from every epoch, which otherwise dominates wall-clock on EyePACS.\n    \"\"\"\n    from concurrent.futures import ThreadPoolExecutor\n    ds = FundusDataset(records, size=size, graham=graham, cache_dir=cache_dir)\n    done = 0\n\n    def one(i):\n        try:\n            ds._load(records[i])\n            return True\n        except Exception as e:\n            print(f\"  skip {records[i].image_path}: {e}\")\n            return False\n\n    with ThreadPoolExecutor(max_workers=workers) as ex:\n        for ok in ex.map(one, range(len(records))):\n            done += bool(ok)\n            if done % 500 == 0:\n                print(f\"  cached {done}/{len(records)}\", flush=True)\n    return done\n",
"dr/train.py": "\"\"\"Train the DR grader on real data.\n\nTypical Kaggle run (P100/T4, ~9h for the full thing):\n\n    python -m dr.train --datasets aptos idrid --size 512 \\\n        --backbone tf_efficientnet_b3_ns --epochs 15 --batch-size 12\n\n    python -m dr.train --datasets eyepacs aptos idrid --size 640 \\\n        --backbone tf_efficientnet_b4_ns --epochs 8 --batch-size 8 \\\n        --cache-dir /kaggle/working/cache\n\nDesign points that matter:\n\n* CHECKPOINT EVERY EPOCH. Kaggle kills sessions at 12h and on idle; a run that\n  cannot resume is a run that never finishes on EyePACS.\n* Thresholds are fitted on VALIDATION predictions, never on training ones, and\n  are saved with the weights. A checkpoint without its cut-points is unusable,\n  because the raw expected-grade scores are calibrated to nothing.\n* Selection is on QWK, not loss. Loss improves while the model gets better at\n  the majority grade; QWK is what the task is actually scored on.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom . import datasets as D, metrics as M, model as MD, splits as S\nfrom .torchdata import build_loaders, build_cache\n\n\ndef parse_args(argv=None):\n    p = argparse.ArgumentParser(description=\"Train the DR grader on real data\")\n    p.add_argument(\"--datasets\", nargs=\"+\", default=[\"aptos\"],\n                   choices=sorted(D.LOADERS), help=\"corpora to train on\")\n    p.add_argument(\"--external\", nargs=\"+\", default=[],\n                   choices=sorted(D.LOADERS),\n                   help=\"corpora held out entirely for external validation\")\n    p.add_argument(\"--data-root\", default=None)\n    p.add_argument(\"--backbone\", default=\"tf_efficientnet_b3_ns\")\n    p.add_argument(\"--size\", type=int, default=512)\n    p.add_argument(\"--epochs\", type=int, default=15)\n    p.add_argument(\"--batch-size\", type=int, default=12)\n    p.add_argument(\"--accum\", type=int, default=1, help=\"gradient accumulation steps\")\n    p.add_argument(\"--lr\", type=float, default=3e-4)\n    p.add_argument(\"--weight-decay\", type=float, default=1e-4)\n    p.add_argument(\"--workers\", type=int, default=2)\n    p.add_argument(\"--val-fraction\", type=float, default=0.2)\n    p.add_argument(\"--seed\", type=int, default=0)\n    p.add_argument(\"--cache-dir\", default=None)\n    p.add_argument(\"--build-cache\", action=\"store_true\",\n                   help=\"materialise the resized cache, then exit\")\n    p.add_argument(\"--out\", default=\"artifacts\")\n    p.add_argument(\"--resume\", default=None)\n    p.add_argument(\"--balance-power\", type=float, default=0.5)\n    p.add_argument(\"--no-pretrained\", action=\"store_true\")\n    p.add_argument(\"--limit\", type=int, default=0, help=\"debug: cap dataset size\")\n    return p.parse_args(argv)\n\n\ndef set_seed(seed):\n    import random\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device, tta=True):\n    \"\"\"Validation pass. Horizontal+vertical flip TTA is cheap and reliably\n    worth ~0.005-0.01 QWK on this task.\"\"\"\n    model.eval()\n    scores, labels = [], []\n    for x, y, _ in loader:\n        x = x.to(device, non_blocking=True)\n        with torch.autocast(device_type=device.type, enabled=device.type == \"cuda\"):\n            logit_sum = model(x).float()\n            n = 1\n            if tta:\n                for dims in ([3], [2], [2, 3]):\n                    logit_sum = logit_sum + model(torch.flip(x, dims=dims)).float()\n                    n += 1\n            logits = logit_sum / n\n        scores.append(M.coral_expected_grade(logits.cpu().numpy()))\n        labels.append(y.numpy())\n    return np.concatenate(scores), np.concatenate(labels)\n\n\ndef main(argv=None):\n    args = parse_args(argv)\n    set_seed(args.seed)\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"device: {device}\"\n          + (f\" ({torch.cuda.get_device_name(0)})\" if device.type == \"cuda\" else \"\"))\n\n    print(\"loading datasets:\")\n    records = D.load(args.datasets, args.data_root)\n    if not records:\n        raise SystemExit(\n            \"No images loaded. Check --data-root / DR_DATA_ROOT, or see the \"\n            \"download instructions printed above.\")\n    if args.limit:\n        records = records[:args.limit]\n    print(D.describe(records))\n\n    if args.build_cache:\n        n = build_cache(records, args.size, True, args.cache_dir, workers=args.workers)\n        print(f\"cached {n} images to {args.cache_dir}\")\n        return\n\n    train_idx, val_idx = S.train_val_split(records, args.val_fraction, args.seed)\n    S.assert_no_patient_leakage(records, train_idx, val_idx)\n    print(\"split (patient-grouped, grade-stratified):\")\n    print(S.summarise_split(records, train_idx, val_idx))\n\n    train_dl, val_dl = build_loaders(\n        records, train_idx, val_idx, args.size, args.batch_size, args.workers,\n        balanced=True, cache_dir=args.cache_dir, seed=args.seed,\n        balance_power=args.balance_power)\n\n    model = MD.build(args.backbone, pretrained=not args.no_pretrained).to(device)\n    criterion = MD.CoralLoss().to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr,\n                            weight_decay=args.weight_decay)\n    steps = max(1, len(train_dl) // args.accum) * args.epochs\n    sched = torch.optim.lr_scheduler.OneCycleLR(\n        opt, max_lr=args.lr, total_steps=steps, pct_start=0.15)\n    scaler = torch.amp.GradScaler(enabled=device.type == \"cuda\")\n\n    start_epoch, best_qwk = 0, -1.0\n    if args.resume and Path(args.resume).exists():\n        ck = torch.load(args.resume, map_location=device)\n        model.load_state_dict(ck[\"model\"])\n        opt.load_state_dict(ck[\"optimizer\"])\n        sched.load_state_dict(ck[\"scheduler\"])\n        scaler.load_state_dict(ck[\"scaler\"])\n        start_epoch = ck[\"epoch\"] + 1\n        best_qwk = ck.get(\"best_qwk\", -1.0)\n        print(f\"resumed from {args.resume} at epoch {start_epoch} (best QWK {best_qwk:.4f})\")\n\n    history = []\n    for epoch in range(start_epoch, args.epochs):\n        model.train()\n        t0, running, seen = time.time(), 0.0, 0\n        opt.zero_grad(set_to_none=True)\n\n        for step, (x, y, _) in enumerate(train_dl):\n            x = x.to(device, non_blocking=True)\n            y = y.to(device, non_blocking=True)\n            with torch.autocast(device_type=device.type, enabled=device.type == \"cuda\"):\n                loss = criterion(model(x), y) / args.accum\n            scaler.scale(loss).backward()\n\n            if (step + 1) % args.accum == 0:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                scaler.step(opt)\n                scaler.update()\n                opt.zero_grad(set_to_none=True)\n                if sched.last_epoch < sched.total_steps - 1:\n                    sched.step()\n\n            running += float(loss.detach()) * args.accum * x.size(0)\n            seen += x.size(0)\n            if step % 50 == 0:\n                print(f\"  epoch {epoch + 1} step {step}/{len(train_dl)} \"\n                      f\"loss {running / max(seen, 1):.4f} \"\n                      f\"lr {sched.get_last_lr()[0]:.2e}\", flush=True)\n\n        val_scores, val_labels = evaluate(model, val_dl, device)\n        # Cut-points are fitted on validation scores only. Fitting them on\n        # training scores would leak and would also be calibrated to a\n        # distribution the model has already overfitted.\n        thresholds, fitted_qwk = M.optimise_thresholds(val_labels, val_scores)\n        preds = M.apply_thresholds(val_scores, thresholds)\n        stats = M.summarise(val_labels, preds)\n\n        print(f\"\\nepoch {epoch + 1}/{args.epochs}  \"\n              f\"train_loss {running / max(seen, 1):.4f}  \"\n              f\"{time.time() - t0:.0f}s\")\n        print(M.format_report(stats, \"validation\"))\n        print(f\"  thresholds: {np.round(thresholds, 3).tolist()}\\n\", flush=True)\n\n        history.append({\"epoch\": epoch + 1, \"train_loss\": running / max(seen, 1),\n                        **{k: v for k, v in stats.items() if k != \"confusion\"}})\n\n        ckpt = {\"model\": model.state_dict(), \"optimizer\": opt.state_dict(),\n                \"scheduler\": sched.state_dict(), \"scaler\": scaler.state_dict(),\n                \"epoch\": epoch, \"best_qwk\": max(best_qwk, stats[\"qwk\"]),\n                \"thresholds\": thresholds.tolist(), \"args\": vars(args),\n                \"backbone\": args.backbone, \"size\": args.size}\n        torch.save(ckpt, out / \"last.pt\")\n\n        # Select on QWK, not on loss: loss keeps improving on the majority\n        # grade long after the clinically useful ranking has stopped improving.\n        if stats[\"qwk\"] > best_qwk:\n            best_qwk = stats[\"qwk\"]\n            torch.save(ckpt, out / \"best.pt\")\n            (out / \"metrics.json\").write_text(json.dumps(\n                {\"val\": stats, \"thresholds\": thresholds.tolist(),\n                 \"epoch\": epoch + 1, \"args\": vars(args)}, indent=2))\n            print(f\"  new best QWK {best_qwk:.4f} -> {out / 'best.pt'}\\n\")\n\n        (out / \"history.json\").write_text(json.dumps(history, indent=2))\n\n    # ------------------------------------------------- external validation\n    if args.external:\n        print(\"\\n=== external validation (never trained on) ===\")\n        ck = torch.load(out / \"best.pt\", map_location=device)\n        model.load_state_dict(ck[\"model\"])\n        th = np.array(ck[\"thresholds\"])\n        ext = D.load(args.external, args.data_root)\n        if ext:\n            print(D.describe(ext))\n            _, ext_dl = build_loaders(ext, [], list(range(len(ext))), args.size,\n                                      args.batch_size, args.workers,\n                                      balanced=False, cache_dir=args.cache_dir)\n            scores, labels = evaluate(model, ext_dl, device)\n            ext_stats = M.summarise(labels, M.apply_thresholds(scores, th))\n            print(M.format_report(ext_stats, \"external\"))\n            (out / \"external_metrics.json\").write_text(json.dumps(ext_stats, indent=2))\n\n    # Export for serving. The ONNX artefact is what the API loads, so the\n    # server never has to import the training code.\n    try:\n        ck = torch.load(out / \"best.pt\", map_location=\"cpu\")\n        model.load_state_dict(ck[\"model\"])\n        MD.export_onnx(model, out / \"grader.onnx\", size=args.size)\n        (out / \"grader.json\").write_text(json.dumps({\n            \"backbone\": args.backbone, \"size\": args.size,\n            \"thresholds\": ck[\"thresholds\"], \"graham\": True,\n            \"val_qwk\": ck.get(\"best_qwk\"), \"datasets\": args.datasets}, indent=2))\n        print(f\"exported {out / 'grader.onnx'}\")\n    except Exception as e:\n        print(f\"ONNX export skipped: {e}\")\n\n    print(f\"\\nbest validation QWK: {best_qwk:.4f}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
"dr/eval_lesions.py": "\"\"\"Score the morphological segmenter against IDRiD's pixel-level lesion masks.\n\n    python -m dr.eval_lesions --data-root /path/to/datasets\n\nIDRiD is the only public corpus that ships per-class lesion masks, which makes\nit the only way to measure the \"where\" channel honestly. APTOS, EyePACS and\nMessidor-2 carry a grade per image and nothing else; a segmenter evaluated\nagainst those can only ever be assessed indirectly.\n\nTwo scores are reported because they answer different questions:\n\n  LESION-LEVEL   did we find each individual lesion? Detections are matched to\n                 connected components of the truth mask within a tolerance.\n                 This is what the clinician sees on the overlay.\n  PIXEL-LEVEL    IoU and Dice against the mask. Harsher, and dominated by\n                 boundary disagreement on large haemorrhages, but it is the\n                 number the segmentation literature reports.\n\nGround truth is mapped through the SAME crop-and-resize the image went\nthrough. Resizing the full original frame instead leaves truth a few percent\nout of register -- invisible by eye, and it silently destroyed recall the last\ntime this evaluation was written.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import defaultdict\n\nimport cv2\nimport numpy as np\n\nfrom core import lesions, preprocess\nfrom . import datasets as D\n\n# Matching tolerance in microns, converted to pixels at the working resolution.\n# 250 um is about the radius of a large microaneurysm -- tight enough that a\n# detection must genuinely overlap the lesion.\nTOLERANCE_UM = 250.0\n\nDARK = (\"MA\", \"HEM\")\nBRIGHT = (\"EX\", \"CWS\")\n\n\ndef load_masks(record, prep):\n    \"\"\"Read IDRiD masks and map them into the prepared image's frame.\"\"\"\n    out = {}\n    for kind, path in record.lesion_masks.items():\n        m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n        if m is None:\n            continue\n        out[kind] = (prep.map_from_original((m > 0).astype(np.uint8)) > 0).astype(np.uint8)\n    return out\n\n\ndef score_record(record, tolerance_px):\n    prep = preprocess.prepare(cv2.imread(record.image_path, cv2.IMREAD_COLOR))\n    truth = load_masks(record, prep)\n    if not truth:\n        return None\n    lmap = lesions.detect(prep)\n\n    kernel = np.ones((tolerance_px * 2 + 1,) * 2, np.uint8)\n    result = {}\n\n    for group, kinds in ((\"dark\", DARK), (\"bright\", BRIGHT)):\n        gt = np.zeros(prep.green.shape, np.uint8)\n        for k in kinds:\n            if k in truth:\n                gt |= truth[k]\n        if not gt.any():\n            continue\n\n        pred = np.zeros(prep.green.shape, np.uint8)\n        points = []\n        for k in kinds:\n            layer = lmap.overlays.get(k)\n            if layer is not None:\n                pred |= layer\n        for l in lmap.lesions:\n            if l.kind in kinds:\n                points.append((l.x, l.y))\n\n        # --- lesion level ---\n        gt_dilated = cv2.dilate(gt, kernel)\n        tp = fp = 0\n        hit_canvas = np.zeros(gt.shape, np.uint8)\n        h, w = gt.shape\n        for x, y in points:\n            if gt_dilated[min(y, h - 1), min(x, w - 1)] > 0:\n                tp += 1\n            else:\n                fp += 1\n            cv2.circle(hit_canvas, (x, y), tolerance_px, 1, -1)\n\n        n, labels, stats, _ = cv2.connectedComponentsWithStats(gt, 8)\n        fn = 0\n        for i in range(1, n):\n            if stats[i, cv2.CC_STAT_AREA] < 3:\n                continue\n            if not (hit_canvas[labels == i] > 0).any():\n                fn += 1\n\n        # --- pixel level ---\n        inter = int((pred & gt).sum())\n        union = int((pred | gt).sum())\n        result[group] = {\n            \"tp\": tp, \"fp\": fp, \"fn\": fn,\n            \"inter\": inter, \"union\": union,\n            \"pred_px\": int(pred.sum()), \"gt_px\": int(gt.sum()),\n        }\n    return result\n\n\ndef aggregate(per_image):\n    totals = defaultdict(lambda: defaultdict(int))\n    for res in per_image:\n        for group, d in res.items():\n            for k, v in d.items():\n                totals[group][k] += v\n\n    report = {}\n    for group, d in totals.items():\n        prec = d[\"tp\"] / max(d[\"tp\"] + d[\"fp\"], 1)\n        rec = d[\"tp\"] / max(d[\"tp\"] + d[\"fn\"], 1)\n        f1 = 2 * prec * rec / max(prec + rec, 1e-9)\n        iou = d[\"inter\"] / max(d[\"union\"], 1)\n        dice = 2 * d[\"inter\"] / max(d[\"pred_px\"] + d[\"gt_px\"], 1)\n        report[group] = {\"precision\": prec, \"recall\": rec, \"f1\": f1,\n                         \"iou\": iou, \"dice\": dice,\n                         \"tp\": d[\"tp\"], \"fp\": d[\"fp\"], \"fn\": d[\"fn\"]}\n    return report\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=\"Lesion segmentation vs IDRiD masks\")\n    ap.add_argument(\"--data-root\", default=None)\n    ap.add_argument(\"--limit\", type=int, default=0)\n    args = ap.parse_args(argv)\n\n    try:\n        records = D.load_idrid_segmentation(args.data_root)\n    except D.DatasetUnavailable as e:\n        print(e)\n        return 1\n    if not records:\n        print(\"IDRiD found, but no images carry lesion masks. Make sure the \"\n              \"'A. Segmentation' folder is present.\")\n        return 1\n    if args.limit:\n        records = records[:args.limit]\n\n    tolerance_px = max(2, int(round(TOLERANCE_UM / (13000.0 / preprocess.TARGET))))\n    print(f\"scoring {len(records)} IDRiD images \"\n          f\"(tolerance {TOLERANCE_UM:.0f} um = {tolerance_px} px)\\n\")\n\n    per_image = []\n    for i, rec in enumerate(records, 1):\n        try:\n            res = score_record(rec, tolerance_px)\n        except Exception as e:                    # noqa: BLE001\n            print(f\"  skip {rec.image_path}: {e}\")\n            continue\n        if res:\n            per_image.append(res)\n        if i % 10 == 0:\n            print(f\"  {i}/{len(records)}\", flush=True)\n\n    if not per_image:\n        print(\"no images produced a score\")\n        return 1\n\n    report = aggregate(per_image)\n    print(f\"\\n{'group':8} {'prec':>7} {'recall':>7} {'F1':>7} {'IoU':>7} {'Dice':>7}\"\n          f\" {'TP':>7} {'FP':>7} {'FN':>7}\")\n    for group in (\"dark\", \"bright\"):\n        if group not in report:\n            continue\n        r = report[group]\n        print(f\"{group:8} {r['precision']:7.3f} {r['recall']:7.3f} {r['f1']:7.3f} \"\n              f\"{r['iou']:7.3f} {r['dice']:7.3f} {r['tp']:7d} {r['fp']:7d} {r['fn']:7d}\")\n    print(f\"\\nscored {len(per_image)} images with masks\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
"core/__init__.py": "",
"core/quality.py": "\"\"\"Image quality gate.\n\nRuns before any analysis. An unusable fundus photo must be rejected at the\npoint of capture, while the patient is still in the chair -- a wrong grade on a\nblurred image is worse than no grade at all.\n\nMATLAB equivalents: var(imfilter(I,fspecial('laplacian'))), stdfilt, im2double.\n\"\"\"\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\n\n# Thresholds are in the units produced by the metric functions below. Sharpness\n# is a mid-band structure ratio x1000: a well-focused fundus image scores ~225,\n# a mildly soft one ~205, and visible blur falls below ~170.\nSHARPNESS_MIN = 155.0\nILLUM_UNIFORMITY_MIN = 0.45\nFOV_COVERAGE_MIN = 0.25\nFOV_COVERAGE_MAX = 0.95\nCLIPPED_HIGHLIGHT_MAX = 0.015\nMEAN_LUMA_MIN, MEAN_LUMA_MAX = 28.0, 215.0\n\n\n@dataclass\nclass QualityReport:\n    passed: bool\n    score: float                      # 0-100, for the ASHA-facing dial\n    metrics: dict = field(default_factory=dict)\n    failures: list = field(default_factory=list)\n    guidance: str = \"\"\n\n    def to_dict(self):\n        return {\n            \"passed\": self.passed,\n            \"score\": round(self.score, 1),\n            \"metrics\": {k: round(float(v), 4) for k, v in self.metrics.items()},\n            \"failures\": self.failures,\n            \"guidance\": self.guidance,\n        }\n\n\ndef field_of_view_mask(bgr):\n    \"\"\"Segment the illuminated retinal circle from the black surround.\"\"\"\n    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n    # Otsu on a blurred copy is robust to the vignette edge being soft.\n    blur = cv2.GaussianBlur(gray, (0, 0), 3)\n    thr = max(10, int(0.5 * np.percentile(blur[blur > 0], 60)) if (blur > 0).any() else 10)\n    mask = (blur > thr).astype(np.uint8)\n    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if n > 1:  # keep the largest blob, discarding reflections and text overlays\n        largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n        mask = (labels == largest).astype(np.uint8)\n    return mask\n\n\ndef _interior(mask, fov_radius):\n    \"\"\"Mask shrunk away from the vignette boundary.\n\n    The FOV rim is the strongest edge in the frame by a wide margin. Left in, it\n    dominates every focus statistic, and a badly blurred retina still scores as\n    sharp because the rim is still a hard edge.\n    \"\"\"\n    r = max(3, int(0.06 * fov_radius))\n    return cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1,) * 2))\n\n\ndef _sharpness(gray, interior, fov_radius):\n    \"\"\"Mid-band structure ratio -- an exposure-invariant focus measure.\n\n    Two approaches fail here and are worth naming, because both are the obvious\n    thing to reach for:\n\n      * Laplacian variance (raw or contrast-normalised) is not monotonic in\n        blur. Normalising by image variance divides out the very signal being\n        measured, so an underexposed frame scores as extremely sharp.\n      * A high-band / mid-band energy ratio inverts. Sensor noise is added\n        AFTER the optical blur in a real camera, so a blurred frame keeps its\n        full noise floor in the high band and scores as sharper than a crisp one.\n\n    Measuring mid-scale structure (2-8 px, the scale of vessels and lesions)\n    against overall retinal contrast avoids both. Blur destroys mid-band\n    structure while leaving low-frequency contrast intact, and exposure scales\n    numerator and denominator together so it cancels.\n    \"\"\"\n    sel = interior > 0\n    if sel.sum() < 64:\n        return 0.0\n    # Light pre-blur suppresses per-pixel sensor noise without touching the\n    # 2-8 px band being measured.\n    g = cv2.GaussianBlur(gray.astype(np.float32), (0, 0), 0.8)\n    scale = max(fov_radius, 1.0) / 256.0\n    mid = (cv2.GaussianBlur(g, (0, 0), 2.0 * scale)\n           - cv2.GaussianBlur(g, (0, 0), 8.0 * scale))\n    contrast = float(g[sel].std())\n    if contrast < 1e-3:\n        return 0.0\n    return float(mid[sel].std()) / contrast * 1000.0\n\n\ndef _illumination_uniformity(gray, mask):\n    \"\"\"1.0 = evenly lit; falls toward 0 as one side of the retina drops into\n    shadow.\n\n    Measures DIRECTIONAL imbalance only. Every fundus image has strong radial\n    falloff toward the periphery -- that is normal optics, not a capture fault,\n    and a plain min/max or decile ratio flags every healthy image because of it.\n    Fitting a plane to the low-frequency luminance isolates the one-sided\n    shadow that actually warrants a recapture.\n    \"\"\"\n    small = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)\n    m = cv2.resize(mask, (64, 64), interpolation=cv2.INTER_NEAREST) > 0\n    if m.sum() < 64:\n        return 0.0\n    lowfreq = cv2.GaussianBlur(small, (0, 0), 6)\n    ys, xs = np.where(m)\n    z = lowfreq[m]\n    mean = float(z.mean())\n    if mean < 1e-6:\n        return 0.0\n    # Least-squares plane z = a*x + b*y + c over the retinal area.\n    A = np.stack([xs - xs.mean(), ys - ys.mean(), np.ones_like(xs)], axis=1).astype(np.float32)\n    coef, *_ = np.linalg.lstsq(A, z, rcond=None)\n    # Peak-to-peak of the fitted tilt across the retina, relative to brightness.\n    span = float(abs(coef[0]) * (xs.max() - xs.min()) + abs(coef[1]) * (ys.max() - ys.min()))\n    return float(np.clip(1.0 - span / mean, 0.0, 1.0))\n\n\ndef assess(bgr) -> QualityReport:\n    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n    h, w = gray.shape\n    mask = field_of_view_mask(bgr)\n    coverage = float(mask.sum()) / (h * w)\n    fov_radius = float(np.sqrt(max(mask.sum(), 1) / np.pi))\n\n    interior = _interior(mask, fov_radius)\n    sharp = _sharpness(gray, interior, fov_radius)\n    uniformity = _illumination_uniformity(gray, mask)\n    inside = gray[interior > 0] if interior.any() else gray[mask > 0]\n    mean_luma = float(inside.mean()) if inside.size else 0.0\n    clipped = float((inside >= 250).mean()) if inside.size else 1.0\n\n    metrics = {\n        \"sharpness\": sharp,\n        \"illumination_uniformity\": uniformity,\n        \"fov_coverage\": coverage,\n        \"mean_luminance\": mean_luma,\n        \"clipped_highlight_fraction\": clipped,\n    }\n\n    failures, guidance = [], []\n    if coverage < FOV_COVERAGE_MIN:\n        failures.append(\"fov_too_small\")\n        guidance.append(\"Retina fills too little of the frame - move the camera closer.\")\n    elif coverage > FOV_COVERAGE_MAX:\n        failures.append(\"fov_cropped\")\n        guidance.append(\"Frame is cropped - pull back so the whole circle is visible.\")\n    # Glare is checked before blur: a saturated blob destroys mid-band structure\n    # and so also trips the blur test, where \"refocus\" would be useless advice.\n    # Clipping with a normal overall exposure is a localised reflection; clipping\n    # with high overall exposure is simply too much flash, and gets the\n    # overexposure message below instead.\n    localised_glare = clipped > CLIPPED_HIGHLIGHT_MAX and mean_luma <= MEAN_LUMA_MAX * 0.8\n    if localised_glare:\n        failures.append(\"glare\")\n        guidance.append(\"Lens glare or reflection detected - tilt the camera \"\n                        \"slightly off-axis and recapture.\")\n    elif sharp < SHARPNESS_MIN:\n        failures.append(\"blurred\")\n        guidance.append(\"Image is blurred - hold steady and refocus, then recapture.\")\n    if uniformity < ILLUM_UNIFORMITY_MIN:\n        failures.append(\"uneven_illumination\")\n        guidance.append(\"One side is in shadow - centre the flash on the pupil.\")\n    if mean_luma < MEAN_LUMA_MIN:\n        failures.append(\"underexposed\")\n        guidance.append(\"Too dark - increase flash or dilate the pupil further.\")\n    elif mean_luma > MEAN_LUMA_MAX or (clipped > CLIPPED_HIGHLIGHT_MAX\n                                       and not localised_glare):\n        failures.append(\"overexposed\")\n        guidance.append(\"Too bright - reduce flash intensity.\")\n\n    # Score blends the normalised sub-metrics; it drives the capture-screen dial.\n    subscores = [\n        min(1.0, sharp / (SHARPNESS_MIN * 1.45)),\n        min(1.0, uniformity / 0.8),\n        1.0 if FOV_COVERAGE_MIN <= coverage <= FOV_COVERAGE_MAX else 0.3,\n        1.0 - min(1.0, clipped / 0.2),\n        1.0 if MEAN_LUMA_MIN <= mean_luma <= MEAN_LUMA_MAX else 0.3,\n    ]\n    score = 100.0 * float(np.mean(subscores))\n\n    return QualityReport(\n        passed=not failures,\n        score=score,\n        metrics=metrics,\n        failures=failures,\n        guidance=\" \".join(guidance) or \"Image quality acceptable for grading.\",\n    )\n",
"core/preprocess.py": "\"\"\"Geometric and photometric normalisation.\n\nEvery downstream stage assumes a square image with the retinal disc centred and\ntangent to the border, so lesion sizes can be reported in microns via a single\nscale factor.\n\nMATLAB equivalents: imcrop, imresize, adapthisteq, imopen.\n\"\"\"\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\nfrom .quality import field_of_view_mask\n\nTARGET = 512\n# A typical 45-degree fundus field spans ~13000 um across the retinal disc.\nFOV_WIDTH_MICRONS = 13000.0\n\n\n@dataclass\nclass Prepared:\n    bgr: np.ndarray        # normalised colour image, TARGET x TARGET\n    green: np.ndarray      # CLAHE-equalised green channel, uint8\n    mask: np.ndarray       # retinal FOV, uint8 {0,1}\n    microns_per_px: float\n    # Geometry of the crop that produced this image, in ORIGINAL image\n    # coordinates: (x0, y0, side). Anything that needs to compare against the\n    # original frame -- ground-truth masks, clinician annotations, a prior\n    # visit's lesion map -- must go through the same transform, or it lands a\n    # few percent off and silently mismatches.\n    crop_origin: tuple = (0, 0)\n    crop_side: int = TARGET\n\n    @property\n    def um2_per_px(self):\n        return self.microns_per_px ** 2\n\n    def map_from_original(self, arr, nearest=True):\n        \"\"\"Bring an original-resolution array into this image's coordinates.\"\"\"\n        x0, y0 = self.crop_origin\n        side = self.crop_side\n        pad = side  # generous, matches the padding used during the crop\n        padded = cv2.copyMakeBorder(arr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)\n        sub = padded[y0 + pad:y0 + pad + side, x0 + pad:x0 + pad + side]\n        interp = cv2.INTER_NEAREST if nearest else cv2.INTER_AREA\n        return cv2.resize(sub, (TARGET, TARGET), interpolation=interp)\n\n    def map_point_from_original(self, x, y):\n        x0, y0 = self.crop_origin\n        s = TARGET / float(self.crop_side)\n        return int(round((x - x0) * s)), int(round((y - y0) * s))\n\n\ndef _crop_to_fov(bgr, mask):\n    \"\"\"Square crop tight to the retinal disc. Returns the crop plus its origin\n    and side in ORIGINAL coordinates, so the transform can be replayed.\"\"\"\n    ys, xs = np.where(mask > 0)\n    if ys.size == 0:\n        return bgr, mask, (0, 0), max(bgr.shape[:2])\n    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1\n    side = int(max(y1 - y0, x1 - x0))\n    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2\n    half = side // 2\n    ox, oy = int(cx - half), int(cy - half)\n    # Pad rather than clamp, so an off-centre retina stays centred after crop.\n    pad = side\n    padded = cv2.copyMakeBorder(bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)\n    pmask = cv2.copyMakeBorder(mask, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)\n    crop = padded[oy + pad:oy + pad + side, ox + pad:ox + pad + side]\n    cmask = pmask[oy + pad:oy + pad + side, ox + pad:ox + pad + side]\n    return crop, cmask, (ox, oy), side\n\n\ndef prepare(bgr) -> Prepared:\n    mask = field_of_view_mask(bgr)\n    crop, cmask, origin, side = _crop_to_fov(bgr, mask)\n    if crop.size == 0:\n        crop, cmask, origin, side = bgr, mask, (0, 0), max(bgr.shape[:2])\n\n    crop = cv2.resize(crop, (TARGET, TARGET), interpolation=cv2.INTER_AREA)\n    cmask = cv2.resize(cmask, (TARGET, TARGET), interpolation=cv2.INTER_NEAREST)\n    # Erode slightly: the vignette boundary otherwise reads as a huge dark lesion.\n    cmask = cv2.erode(cmask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))\n\n    green = crop[:, :, 1]\n    green = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(green)\n    green = cv2.bitwise_and(green, green, mask=cmask)\n\n    microns_per_px = FOV_WIDTH_MICRONS / TARGET\n    return Prepared(bgr=crop, green=green, mask=cmask,\n                    microns_per_px=microns_per_px,\n                    crop_origin=origin, crop_side=side)\n\n\ndef fill_outside_fov(green, mask):\n    \"\"\"Replace the black surround with retinal-like intensity.\n\n    Without this, every morphological operator sees a step edge of ~150 grey\n    levels at the FOV rim and returns a response far larger than any lesion,\n    which swamps the detector's dynamic range. Mirror-style inpainting makes the\n    rim morphologically invisible.\n    \"\"\"\n    inside = green[mask > 0]\n    fill = float(np.median(inside)) if inside.size else 0.0\n    out = green.astype(np.float32)\n    out[mask == 0] = fill\n    # Smooth across the seam so the fill does not itself create an edge.\n    blurred = cv2.GaussianBlur(out, (0, 0), 6)\n    seam = cv2.dilate(1 - mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13)))\n    seam = (seam > 0) & (mask > 0)\n    out[mask == 0] = blurred[mask == 0]\n    out[seam] = blurred[seam]\n    return np.clip(out, 0, 255).astype(np.uint8)\n\n\ndef flatten_illumination(green, mask, sigma=25):\n    \"\"\"Remove the slow background gradient so top-hat responses are comparable\n    between the bright posterior pole and the dim periphery.\"\"\"\n    filled = fill_outside_fov(green, mask).astype(np.float32)\n    bg = cv2.GaussianBlur(filled, (0, 0), sigma)\n    ref = float(np.median(filled[mask > 0])) if (mask > 0).any() else 0.0\n    return np.clip(filled - bg + ref, 0, 255).astype(np.uint8)\n",
"core/lesions.py": "\"\"\"Multi-scale morphological lesion segmentation -- the 'where' channel.\n\nThis stage is deliberately NOT a neural network. It is deterministic,\ninspectable classical image processing, so every lesion the system reports can\nbe traced back to a specific morphological response at a specific scale. That\nis what lets a clinician audit the machine rather than trust it.\n\nLesion classes follow the ICDR grading vocabulary:\n  MA  microaneurysm       small round dark, under 125 um\n  HEM haemorrhage         larger irregular dark\n  EX  hard exudate        bright, sharp-edged lipid deposit\n  CWS cotton-wool spot    bright, fuzzy-edged nerve-fibre infarct\n\nMATLAB equivalents: imtophat / imbothat with strel disk, imreconstruct,\nfibermetric, and regionprops for Area, Circularity, Centroid, MajorAxisLength.\n\"\"\"\nfrom dataclasses import dataclass, asdict\n\nimport cv2\nimport numpy as np\nfrom skimage.filters import frangi\n\nfrom .preprocess import flatten_illumination\n\n# Radii in pixels at the 512px working resolution (~25 um/px).\nDARK_SCALES = (2, 4, 7, 11)\nBRIGHT_SCALES = (3, 6, 10)\nMA_MAX_DIAMETER_UM = 125.0\n\n# Anatomical constants as a fraction of the field-of-view width. The fovea sits\n# about 2.5 disc diameters temporal to the optic disc; the band is widened to\n# cover normal inter-patient variation. Expressing these against the FOV rather\n# than against the ESTIMATED disc radius matters -- the radius estimate carries\n# its own error, and compounding it moved the search annulus off the macula\n# entirely.\nDISC_RADIUS_FRAC = 0.09\nMACULA_DIST_FRAC = (0.20, 0.42)\nMACULA_TEMPORAL_COS = 0.77          # accept within ~40 degrees of temporal\n# Mean edge-gradient above which a bright lesion is called a hard exudate.\nEXUDATE_EDGE_SHARPNESS = 90.0\n\n# Detection thresholds, expressed as multiples of the robust noise sigma (MAD)\n# of the top-hat response, with an absolute floor in grey levels.\nDARK_K, DARK_FLOOR, DARK_MIN_PX = 5.0, 10.0, 2\nBRIGHT_K, BRIGHT_FLOOR, BRIGHT_MIN_PX = 13.0, 14.0, 6\nVESSEL_DILATE = 2\nDARK_OPEN_RADIUS = 0     # microaneurysms are only a few px across\nBRIGHT_OPEN_RADIUS = 1   # exudates are larger; opening trims mottling\n\n# A dark component is discarded as a vessel fragment only if it is BOTH mostly\n# covered by the vessel map AND not compact. Deleting every vessel pixel\n# outright removes ~40% of genuine microaneurysms, which sit on the capillary\n# bed by definition; shape is what separates a lesion from a vessel segment.\n# The circularity value is a minimum-enclosing-circle fill ratio (see\n# _shape_stats), so it is tuned against that scale, not against 4*pi*A/P^2.\nVESSEL_OVERLAP_REJECT = 0.70\nVESSEL_CIRCULARITY_KEEP = 0.85\n\n\n@dataclass\nclass Lesion:\n    kind: str\n    x: int\n    y: int\n    area_um2: float\n    major_axis_um: float\n    minor_axis_um: float\n    circularity: float\n    contrast: float\n    dist_to_macula_um: float\n\n    def to_dict(self):\n        d = asdict(self)\n        for k in (\"area_um2\", \"major_axis_um\", \"minor_axis_um\", \"circularity\",\n                  \"contrast\", \"dist_to_macula_um\"):\n            d[k] = round(float(d[k]), 2)\n        return d\n\n\n@dataclass\nclass LesionMap:\n    lesions: list\n    vessels: np.ndarray\n    optic_disc: tuple          # (x, y, radius_px)\n    macula: tuple              # (x, y)\n    overlays: dict             # kind -> binary uint8 mask\n\n    def counts(self):\n        c = {\"MA\": 0, \"HEM\": 0, \"EX\": 0, \"CWS\": 0}\n        for l in self.lesions:\n            c[l.kind] += 1\n        return c\n\n    def burden(self):\n        \"\"\"Total lesion area per class, in square microns.\"\"\"\n        b = {\"MA\": 0.0, \"HEM\": 0.0, \"EX\": 0.0, \"CWS\": 0.0}\n        for l in self.lesions:\n            b[l.kind] += l.area_um2\n        return b\n\n    def macula_involved(self, radius_um=1500.0):\n        \"\"\"Lesions inside the fovea-centred circle drive clinically significant\n        macular oedema risk, which escalates referral urgency independent of\n        the overall severity grade.\"\"\"\n        return [l for l in self.lesions if l.dist_to_macula_um <= radius_um]\n\n\ndef _strel(r):\n    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))\n\n\ndef segment_vessels(green, mask):\n    \"\"\"Frangi vesselness. Vessels are dark and elongated, so they alias into the\n    dark-lesion top-hat; they must be removed before haemorrhage counting.\"\"\"\n    inv = 255 - green\n    v = frangi(inv.astype(np.float32) / 255.0, sigmas=range(1, 6),\n               black_ridges=False)\n    if v.max() > 0:\n        v = v / v.max()\n    ves = (v > 0.08).astype(np.uint8)\n    ves = cv2.morphologyEx(ves, cv2.MORPH_CLOSE, _strel(2))\n    return cv2.bitwise_and(ves, ves, mask=mask)\n\n\ndef locate_optic_disc(bgr, mask):\n    \"\"\"Brightest disc-sized region in the red channel.\n\n    The smoothing scale must match the optic disc, not a lesion. Blurring at a\n    small sigma leaves a tight cluster of hard exudates brighter than the disc\n    itself, and the detector then places the disc on the lesions -- which in\n    turn drives the macula estimate, the exclusion zone and every\n    distance-to-fovea measurement off the same error.\n    \"\"\"\n    radius = int(DISC_RADIUS_FRAC * max(mask.shape))\n    red = cv2.bitwise_and(bgr[:, :, 2], bgr[:, :, 2], mask=mask).astype(np.float32)\n    m = mask.astype(np.float32)\n\n    def smooth_at(sigma):\n        # Normalised convolution. Blurring the masked image alone averages the\n        # black surround into every pixel near the rim, dimming it -- and the\n        # optic disc usually sits near the rim, so a plain blur hides the very\n        # thing being looked for.\n        num = cv2.GaussianBlur(red, (0, 0), sigma)\n        den = cv2.GaussianBlur(m, (0, 0), sigma)\n        return np.where(den > 1e-3, num / np.maximum(den, 1e-3), 0.0)\n\n    # Absolute brightness is the wrong signal: a healthy retina is brightest at\n    # the posterior pole, so the global maximum lands mid-frame and the disc is\n    # missed in almost half of images. What identifies the disc is being bright\n    # RELATIVE TO ITS SURROUNDINGS at its own scale, so the broad background is\n    # subtracted first.\n    response = smooth_at(radius * 0.6) - smooth_at(radius * 3.0)\n    response[mask == 0] = -1e9\n    _, _, _, loc = cv2.minMaxLoc(response)\n    return (int(loc[0]), int(loc[1]), radius)\n\n\ndef locate_macula(green, mask, disc):\n    \"\"\"Darkest region on the temporal side of the disc, ~2-3 disc diameters out.\n\n    Direction is as important as distance. Searching the whole annulus and\n    taking the darkest pixel finds the vignetted periphery every time -- the\n    edge of the retina is far darker than the fovea -- which put the macula\n    ~260 px from truth and corrupted every distance-to-fovea measurement.\n\n    The fovea lies temporal to the disc, i.e. toward the centre of the frame,\n    and close to the same height. Constraining the search to a wedge in that\n    direction is what makes the estimate track the real anatomy.\n    \"\"\"\n    h, w = green.shape\n    dx, dy, dr = disc\n    smooth = cv2.GaussianBlur(green.astype(np.float32), (0, 0), 14)\n    yy, xx = np.mgrid[0:h, 0:w]\n    vx, vy = (xx - dx).astype(np.float32), (yy - dy).astype(np.float32)\n    dist = np.sqrt(vx ** 2 + vy ** 2) + 1e-6\n\n    # Unit vector from the disc toward the frame centre = the temporal direction.\n    tx, ty = (w / 2.0 - dx), (h / 2.0 - dy)\n    tnorm = float(np.hypot(tx, ty))\n    if tnorm < 1e-6:\n        tx, ty, tnorm = 1.0, 0.0, 1.0\n    cos = (vx * (tx / tnorm) + vy * (ty / tnorm)) / dist\n\n    lo, hi = (f * max(h, w) for f in MACULA_DIST_FRAC)\n    band = ((dist > lo) & (dist < hi)\n            & (cos > MACULA_TEMPORAL_COS)\n            & (mask > 0))\n    if not band.any():\n        # Fall back to the expected anatomical position rather than to the\n        # frame centre, which could sit anywhere relative to the disc.\n        step = 0.5 * sum(MACULA_DIST_FRAC) * max(h, w)\n        fx = int(np.clip(dx + step * tx / tnorm, 0, w - 1))\n        fy = int(np.clip(dy + step * ty / tnorm, 0, h - 1))\n        return (fx, fy)\n    cand = np.where(band, smooth, np.inf)\n    idx = int(np.argmin(cand))\n    return (idx % w, idx // w)\n\n\ndef _multiscale_tophat(img, scales, dark=True):\n    \"\"\"Keep the strongest response across scales, so both a 50 um microaneurysm\n    and a 400 um blot haemorrhage survive the same pass.\"\"\"\n    op = cv2.MORPH_BLACKHAT if dark else cv2.MORPH_TOPHAT\n    acc = np.zeros(img.shape, np.float32)\n    for r in scales:\n        th = cv2.morphologyEx(img, op, _strel(r)).astype(np.float32)\n        acc = np.maximum(acc, th)\n    return acc\n\n\ndef _shape_stats(comp_mask):\n    \"\"\"Return (circularity, major_axis_px, minor_axis_px).\n\n    Circularity is the fraction of the minimum enclosing circle that the\n    component actually fills, clamped to [0, 1]. The textbook 4*pi*A/P^2 is not\n    usable here: microaneurysms are only a few pixels across, and at that size\n    the discrete perimeter is so short that the ratio explodes -- a 2-pixel blob\n    scores pi, well above the 1.0 a perfect disc should give. Since circularity\n    is what separates a microaneurysm from a haemorrhage, that turned every\n    degenerate speck into a confident microaneurysm.\n\n    The fill ratio degrades gracefully instead: a compact blob approaches 1,\n    an elongated vessel fragment falls toward 0.2-0.4, and both stay bounded\n    at any size.\n    \"\"\"\n    cnts, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    if not cnts:\n        return 0.0, 0.0, 0.0\n    c = max(cnts, key=cv2.contourArea)\n    area_px = float(comp_mask.sum())\n\n    (_, _), radius = cv2.minEnclosingCircle(c)\n    circle_area = np.pi * max(radius, 0.5) ** 2\n    circularity = float(np.clip(area_px / circle_area, 0.0, 1.0))\n\n    if len(c) >= 5:\n        (_, _), (ax1, ax2), _ = cv2.fitEllipse(c)\n        major, minor = max(ax1, ax2), min(ax1, ax2)\n        # fitEllipse can return a degenerate zero axis on collinear points.\n        if minor < 1e-3:\n            major = minor = float(np.sqrt(max(area_px, 1.0)))\n    else:\n        major = minor = float(np.sqrt(max(area_px, 1.0)))\n    return circularity, float(major), float(minor)\n\n\ndef _threshold(response, region, k, floor):\n    \"\"\"Robust absolute threshold: median + k * MAD of the top-hat response.\n\n    A percentile threshold would be wrong here -- it declares a fixed fraction\n    of pixels to be lesions no matter what, so a perfectly healthy retina still\n    returns a full lesion inventory. Anchoring to the noise floor instead lets\n    a grade-0 eye legitimately return zero detections.\n    \"\"\"\n    vals = response[region > 0].astype(np.float32)\n    if vals.size == 0:\n        return np.zeros(response.shape, np.uint8)\n    med = float(np.median(vals))\n    mad = float(np.median(np.abs(vals - med))) * 1.4826   # -> sigma equivalent\n    thr = max(med + k * max(mad, 1.0), floor)\n    return (response > thr).astype(np.uint8)\n\n\ndef detect(prep) -> LesionMap:\n    green, mask, upp = prep.green, prep.mask, prep.microns_per_px\n    flat = flatten_illumination(green, mask)\n    vessels = segment_vessels(green, mask)\n    disc = locate_optic_disc(prep.bgr, mask)\n    macula = locate_macula(green, mask, disc)\n\n    disc_mask = np.zeros_like(mask)\n    cv2.circle(disc_mask, (disc[0], disc[1]), int(disc[2] * 1.4), 1, -1)\n    # Keep clear of the rim by more than the largest structuring element, so no\n    # detection is an artefact of the field-of-view boundary.\n    margin = max(DARK_SCALES + BRIGHT_SCALES) + 4\n    interior = cv2.erode(mask, _strel(margin))\n    valid = cv2.bitwise_and(interior, 1 - disc_mask)\n    # Frangi under-covers vessel edges, so the map is dilated before it is used\n    # to judge overlap -- but it is used to JUDGE, not to erase. See below.\n    vessel_wide = cv2.dilate(vessels, _strel(VESSEL_DILATE))\n\n    lesions = []\n    layers = {k: np.zeros(mask.shape, np.uint8) for k in (\"MA\", \"HEM\", \"EX\", \"CWS\")}\n\n    def dist_to_macula(cx, cy):\n        return float(np.hypot(cx - macula[0], cy - macula[1]) * upp)\n\n    # ---- dark lesions: microaneurysms and haemorrhages ----\n    dark = _multiscale_tophat(flat, DARK_SCALES, dark=True)\n    dark = cv2.bitwise_and(dark, dark, mask=valid)\n    # Threshold statistics are computed off-vessel so the vessel population does\n    # not inflate the noise estimate, but detection itself runs over all of the\n    # valid retina.\n    off_vessel = cv2.bitwise_and(valid, 1 - vessel_wide)\n    dbin = _threshold(dark, off_vessel, DARK_K, DARK_FLOOR)\n    dbin = cv2.bitwise_and(dbin, valid)\n    if DARK_OPEN_RADIUS > 0:\n        dbin = cv2.morphologyEx(dbin, cv2.MORPH_OPEN, _strel(DARK_OPEN_RADIUS))\n\n    n, labels, stats, cents = cv2.connectedComponentsWithStats(dbin, 8)\n    for i in range(1, n):\n        area_px = int(stats[i, cv2.CC_STAT_AREA])\n        if area_px < DARK_MIN_PX:\n            continue\n        comp = (labels == i).astype(np.uint8)\n        circ, major, minor = _shape_stats(comp)\n        # Vessel-fragment rejection: only discard things that are both sitting\n        # on a vessel and shaped like one.\n        overlap = float((vessel_wide[comp > 0] > 0).mean())\n        if overlap > VESSEL_OVERLAP_REJECT and circ < VESSEL_CIRCULARITY_KEEP:\n            continue\n        major_um = major * upp\n        cx, cy = int(cents[i][0]), int(cents[i][1])\n        # ICDR: a microaneurysm is round and under ~125 um across; anything\n        # larger or irregular is counted as a haemorrhage.\n        kind = \"MA\" if (major_um <= MA_MAX_DIAMETER_UM and circ > 0.55) else \"HEM\"\n        layers[kind][labels == i] = 1\n        lesions.append(Lesion(\n            kind=kind, x=cx, y=cy,\n            area_um2=area_px * prep.um2_per_px,\n            major_axis_um=major_um, minor_axis_um=minor * upp,\n            circularity=circ, contrast=float(dark[comp > 0].mean()),\n            dist_to_macula_um=dist_to_macula(cx, cy),\n        ))\n\n    # ---- bright lesions: hard exudates and cotton-wool spots ----\n    bright = _multiscale_tophat(flat, BRIGHT_SCALES, dark=False)\n    bright = cv2.bitwise_and(bright, bright, mask=valid)\n    bbin = _threshold(bright, valid, BRIGHT_K, BRIGHT_FLOOR)\n    bbin = cv2.morphologyEx(bbin, cv2.MORPH_OPEN, _strel(BRIGHT_OPEN_RADIUS))\n    # Edge sharpness separates lipid exudates (crisp) from CWS infarcts (fuzzy).\n    gx, gy = cv2.spatialGradient(flat)\n    grad = cv2.magnitude(gx.astype(np.float32), gy.astype(np.float32))\n\n    n, labels, stats, cents = cv2.connectedComponentsWithStats(bbin, 8)\n    for i in range(1, n):\n        area_px = int(stats[i, cv2.CC_STAT_AREA])\n        if area_px < BRIGHT_MIN_PX:\n            continue\n        comp = (labels == i).astype(np.uint8)\n        circ, major, minor = _shape_stats(comp)\n        edge = cv2.dilate(comp, _strel(1)) - cv2.erode(comp, _strel(1))\n        sharpness = float(grad[edge > 0].mean()) if (edge > 0).any() else 0.0\n        cx, cy = int(cents[i][0]), int(cents[i][1])\n        kind = \"EX\" if sharpness > EXUDATE_EDGE_SHARPNESS else \"CWS\"\n        layers[kind][labels == i] = 1\n        lesions.append(Lesion(\n            kind=kind, x=cx, y=cy,\n            area_um2=area_px * prep.um2_per_px,\n            major_axis_um=major * upp, minor_axis_um=minor * upp,\n            circularity=circ, contrast=float(bright[comp > 0].mean()),\n            dist_to_macula_um=dist_to_macula(cx, cy),\n        ))\n\n    overlays = dict(layers)\n    overlays[\"vessels\"] = vessels\n    return LesionMap(lesions=lesions, vessels=vessels, optic_disc=disc,\n                     macula=macula, overlays=overlays)\n",
"core/features.py": "\"\"\"Lesion inventory -> fixed-length clinical feature vector.\n\nEvery feature here is a quantity an ophthalmologist would recognise and could\ncheck by hand. That is the point: the feature-based grader is not a fallback\nfor want of a GPU, it is the auditable half of the system. When the CNN and\nthis model disagree, the disagreement itself is reportable.\n\"\"\"\nimport numpy as np\n\n# Order is fixed and used for both training and explanation.\nFEATURE_NAMES = [\n    \"ma_count\", \"hem_count\", \"ex_count\", \"cws_count\",\n    \"ma_area_frac\", \"hem_area_frac\", \"ex_area_frac\", \"cws_area_frac\",\n    \"dark_count\", \"bright_count\", \"total_count\",\n    \"ma_mean_size_um\", \"hem_mean_size_um\", \"ex_mean_size_um\",\n    \"hem_max_size_um\", \"ex_max_size_um\",\n    \"macula_lesion_count\", \"macula_ex_count\", \"min_ex_dist_to_macula_um\",\n    \"quadrants_with_hem\", \"quadrants_with_ma\",\n    \"vessel_density\", \"lesion_spatial_spread\",\n]\n\n# Readable labels for the clinician-facing explanation panel.\nFEATURE_LABELS = {\n    \"ma_count\": \"microaneurysms\",\n    \"hem_count\": \"haemorrhages\",\n    \"ex_count\": \"hard exudates\",\n    \"cws_count\": \"cotton-wool spots\",\n    \"quadrants_with_hem\": \"retinal quadrants containing haemorrhage\",\n    \"quadrants_with_ma\": \"retinal quadrants containing microaneurysms\",\n    \"macula_ex_count\": \"exudates within 1500 um of the fovea\",\n    \"min_ex_dist_to_macula_um\": \"closest exudate to the fovea\",\n    \"hem_max_size_um\": \"largest haemorrhage\",\n    \"vessel_density\": \"visible vessel density\",\n}\n\nRETINA_AREA_UM2 = np.pi * (13000.0 / 2) ** 2\n\n\ndef _quadrant(l, centre):\n    return (0 if l.x < centre[0] else 1) + (0 if l.y < centre[1] else 2)\n\n\ndef extract(lesion_map, prep):\n    by = {\"MA\": [], \"HEM\": [], \"EX\": [], \"CWS\": []}\n    for l in lesion_map.lesions:\n        by[l.kind].append(l)\n    counts = {k: len(v) for k, v in by.items()}\n    burden = lesion_map.burden()\n\n    def mean_size(k):\n        return float(np.mean([l.major_axis_um for l in by[k]])) if by[k] else 0.0\n\n    def max_size(k):\n        return float(np.max([l.major_axis_um for l in by[k]])) if by[k] else 0.0\n\n    centre = (prep.green.shape[1] // 2, prep.green.shape[0] // 2)\n    quad_hem = len({_quadrant(l, centre) for l in by[\"HEM\"]})\n    quad_ma = len({_quadrant(l, centre) for l in by[\"MA\"]})\n\n    macula_lesions = lesion_map.macula_involved()\n    macula_ex = [l for l in macula_lesions if l.kind == \"EX\"]\n    min_ex_dist = (min((l.dist_to_macula_um for l in by[\"EX\"]), default=13000.0))\n\n    vessel_density = float(lesion_map.vessels.sum()) / max(float(prep.mask.sum()), 1.0)\n\n    if lesion_map.lesions:\n        pts = np.array([[l.x, l.y] for l in lesion_map.lesions], np.float32)\n        spread = float(np.mean(np.std(pts, axis=0))) * prep.microns_per_px\n    else:\n        spread = 0.0\n\n    values = {\n        \"ma_count\": counts[\"MA\"],\n        \"hem_count\": counts[\"HEM\"],\n        \"ex_count\": counts[\"EX\"],\n        \"cws_count\": counts[\"CWS\"],\n        \"ma_area_frac\": burden[\"MA\"] / RETINA_AREA_UM2,\n        \"hem_area_frac\": burden[\"HEM\"] / RETINA_AREA_UM2,\n        \"ex_area_frac\": burden[\"EX\"] / RETINA_AREA_UM2,\n        \"cws_area_frac\": burden[\"CWS\"] / RETINA_AREA_UM2,\n        \"dark_count\": counts[\"MA\"] + counts[\"HEM\"],\n        \"bright_count\": counts[\"EX\"] + counts[\"CWS\"],\n        \"total_count\": sum(counts.values()),\n        \"ma_mean_size_um\": mean_size(\"MA\"),\n        \"hem_mean_size_um\": mean_size(\"HEM\"),\n        \"ex_mean_size_um\": mean_size(\"EX\"),\n        \"hem_max_size_um\": max_size(\"HEM\"),\n        \"ex_max_size_um\": max_size(\"EX\"),\n        \"macula_lesion_count\": len(macula_lesions),\n        \"macula_ex_count\": len(macula_ex),\n        \"min_ex_dist_to_macula_um\": min_ex_dist,\n        \"quadrants_with_hem\": quad_hem,\n        \"quadrants_with_ma\": quad_ma,\n        \"vessel_density\": vessel_density,\n        \"lesion_spatial_spread\": spread,\n    }\n    vector = np.array([float(values[n]) for n in FEATURE_NAMES], np.float32)\n    return vector, values\n"
}""")

for rel, text in _SOURCES.items():
    dest = PROJECT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(text, encoding="utf-8")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

print(f"wrote {len(_SOURCES)} modules to {PROJECT}")
print("  " + "\n  ".join(sorted(_SOURCES)))

## 3 · What the data looks like

Check the corpora resolve, and look hard at the grade distribution. These
datasets run roughly 73% grade 0 and under 3% grade 4, and that imbalance — not
model capacity — is the main reason DR models miss sight-threatening disease.

In [ ]:
import dr.datasets as D

DATASETS = ["aptos"]          # add "idrid", "eyepacs" once attached

records = D.load(DATASETS)
if not records:
    raise SystemExit("No images found. Attach a dataset in the right-hand panel.")
print()
print(D.describe(records))

## 4 · Split

Grouped by patient, stratified by grade. Both matter:

* **Grouping** — EyePACS carries both eyes of a patient and the two are highly
  correlated. A random *image* split puts one eye in train and the other in
  validation, so the score is partly memorisation. The assertion below fails
  the run rather than reporting an inflated number.
* **Stratification** — grades 3 and 4 are a few percent of these corpora. An
  unstratified fold can contain almost no severe disease, which makes its
  sensitivity estimate meaningless.

In [ ]:
import dr.splits as S

train_idx, val_idx = S.train_val_split(records, val_fraction=0.2, seed=0)
S.assert_no_patient_leakage(records, train_idx, val_idx)
print("no patient appears in both sides\n")
print(S.summarise_split(records, train_idx, val_idx))

## 5 · Cache the resized images  *(optional, do it once)*

Decoding multi-megapixel JPEGs — not the GPU — is what makes an epoch slow. On
EyePACS this is the difference between hours and minutes per epoch. Skip it for
a small APTOS-only run; it costs more than it saves there.

In [ ]:
# from dr.torchdata import build_cache
# build_cache(records, size=512, cache_dir="/kaggle/working/cache", workers=4)

## 6 · Train

**`--pretrained` is not optional.** From random initialisation on a cohort this
size the network collapses to predicting one class for every image: 20%
accuracy, and *worse than having no CNN at all*, because fusion would drag every
grade toward that class while Grad-CAM produced convincing-looking saliency that
meant nothing. The trainer refuses to save such a model.

**Kaggle kills sessions at 12 hours.** Every epoch checkpoints to
`artifacts/last.pt`. To continue, re-run this cell with `--resume` uncommented.

Rough timings on a P100 with APTOS (~3,660 images):

| Backbone | Size | Batch | Per epoch |
|---|---|---|---|
| `tf_efficientnet_b0_ns` | 384 | 24 | ~3 min |
| `tf_efficientnet_b3_ns` | 512 | 12 | ~6 min |
| `tf_efficientnet_b4_ns` | 640 | 8 | ~12 min |

Resolution matters more than depth here — microaneurysms are only a few pixels
across, so dropping below 384 px removes the earliest sign of disease from the
image entirely.

In [ ]:
from dr.train import main as train_main

ARTIFACTS = "/kaggle/working/artifacts"

train_main([
    "--datasets", *DATASETS,
    "--size", "512",
    "--backbone", "tf_efficientnet_b3_ns",
    "--epochs", "12",
    "--batch-size", "12",
    "--lr", "3e-4",
    "--workers", "2",
    "--out", ARTIFACTS,
    # "--cache-dir", "/kaggle/working/cache",
    # "--external", "messidor2",              # never trained on — a true external score
    # "--resume", f"{ARTIFACTS}/last.pt",     # uncomment to continue a killed session
])

## 7 · Results

Read **QWK first**. It is what this task is scored on, and the only common
metric that understands the grades are ordinal — confusing grade 0 with grade 4
is far worse than confusing 3 with 4, and plain accuracy scores those
identically.

Then read **referable sensitivity**: that is what a screening programme is
actually accountable for. The NHS DR screening standard is ≥85% sensitivity and
≥80% specificity for referable disease.

In [ ]:
import json
from pathlib import Path
import dr.metrics as M

art = Path(ARTIFACTS)
metrics = json.loads((art / "metrics.json").read_text())

print(M.format_report(metrics["val"], "validation"))
print()
print("grade cut-points:", [round(t, 3) for t in metrics["thresholds"]])
print("best epoch      :", metrics["epoch"])

In [ ]:
history = json.loads((art / "history.json").read_text())
print(f"{'epoch':>5} {'loss':>9} {'QWK':>8} {'acc':>7} {'ref sens':>9} {'ref spec':>9}")
for h in history:
    print(f"{h['epoch']:>5} {h['train_loss']:>9.4f} {h['qwk']:>8.4f} "
          f"{h['accuracy']:>7.3f} {h['referable_sensitivity']:>9.3f} "
          f"{h['referable_specificity']:>9.3f}")

## 8 · Lesion benchmark  *(IDRiD only)*

IDRiD is the only public corpus with **pixel-level lesion masks**, which makes
it the only way to score the morphological segmenter's "where" channel
honestly. Grade-only corpora can assess it indirectly at best.

Skip this cell if IDRiD is not attached.

In [ ]:
# from dr.eval_lesions import main as eval_lesions
# eval_lesions([])

## 9 · Export and download

`grader.onnx` plus `grader.json` (cut-points, backbone, input size) is
everything the server needs — it loads the frozen graph and never imports the
training code.

Download both from the notebook's **Output** tab, drop them into `artifacts/`
beside the deployment, and restart. `/api/health` will then report the model as
available and grading will stop returning 503.

In [ ]:
for f in sorted(art.iterdir()):
    print(f"{f.name:22} {f.stat().st_size/1e6:9.1f} MB")

In [ ]:
# Parity check: the exported graph must agree with the checkpoint that was
# validated. If these diverge, the model you serve is not the model you
# measured — which would make every number above meaningless.
import numpy as np, torch
import dr.model as MD, dr.metrics as M

ck = torch.load(art / "best.pt", map_location="cpu")
net = MD.build(ck["backbone"], pretrained=False)
net.load_state_dict(ck["model"])
net.eval()

x = np.random.randn(2, 3, ck["size"], ck["size"]).astype(np.float32)
with torch.no_grad():
    torch_grade = M.coral_expected_grade(net(torch.from_numpy(x)).numpy())
onnx_grade, _ = MD.OnnxGrader(art / "grader.onnx")(x)

drift = float(np.abs(torch_grade - onnx_grade).max())
print(f"max |torch - onnx| = {drift:.2e}")
assert drift < 1e-4, "exported graph diverged from the validated checkpoint"
print("parity OK — safe to deploy")

---

## If something goes wrong

**`No images found`** — the dataset is not attached, or its folder name differs
from what the loader expects. Check what is actually mounted:

```python
from pathlib import Path
for p in sorted(Path("/kaggle/input").iterdir()):
    print(p.name, "→", [c.name for c in p.iterdir()][:6])
```

Then pass `--data-root` explicitly, or tell me the layout and I will widen the
loader's candidate paths.

**`REFUSED TO SAVE: model predicts only N distinct grades`** — working as
intended. The model collapsed. Use `--pretrained` (it is on by default; make
sure internet is enabled so the weights can download), or train on more data.

**CUDA out of memory** — halve `--batch-size` and add `--accum 2`, which keeps
the effective batch size while holding half as much in memory.

**Session killed mid-run** — uncomment `--resume` in the training cell. It picks
up from the last completed epoch, optimiser and scheduler state included.